In [ ]:
import os
import sys

# to setup import paths add project root dirs to sys.path
sys.path.append(os.path.join(os.getcwd(), "..", ".."))
from baseVR.base_functionality import init_import_paths
init_import_paths()

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib widget
from matplotlib.colors import Normalize

from analytics_processing import analytics
import analytics_processing.analytics_constants as C
from CustomLogger import CustomLogger as Logger

from analytics_processing.sessions_from_nas_parsing import sessionlist_fullfnames_from_args
from analytics_processing.sessions_from_nas_parsing import fullfnames2snames

import scipy.stats as stats

In [ ]:
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', 100)     # Show all rows
pd.set_option('display.width', 500)        # Wide display
pd.set_option('display.max_colwidth', 40) # Full width column content
pd.set_option('display.expand_frame_repr', False)  # Don't wrap to multiple lines

In [ ]:
# first set the paths and logger. You can see all kind of outputs in this directory 
# already. Poster used plots from this directory.

data = {}
nas_dir = C.device_paths()[0]
output_dir = f"{nas_dir}/Simon/poster/cue_decoding/"
Logger().init_logger(None, None, logging_level="WARNING")

In [ ]:
# analysis was intially done on all sessions from animal 6 with paradigm 1100
# But the poster focused on S8-15

animal_ids = [6]
paradigm_ids = [1100]
session_ids = None

In [ ]:
session_dirs, _ = sessionlist_fullfnames_from_args(paradigm_ids, animal_ids, session_ids)
session_names = fullfnames2snames(session_dirs)

In [ ]:
# this is a new analytic. It is a crucial component that is required by the SVM
# decoding analytic to be computed. It is also used here for plotting.
# Another input analytic that is used to calc TrialWiseT0Events40ms that I use 
# now a lot is Behavior40msAligned - this is similar to BehaviorTrackwise, but 
# instead of binning according to track position, we bin according to the 40ms 
# ephys bins. So usually an average over 2 or 3 unity frames.

# Each row is an interval - a start and a stop ephys time. Think of this as the 
# selecter of firingrates X for the SVM that we want to train with. There are 4 
# differnent intervals a row may define (and 2 more specifcal-case ones): 
# cue_entry_interval, cue_exit_interval, R1_entry_interval, R2_entry_interval
# (cue_exit_interval is not used, in reality it overlaps a lot with R1_entry_interval).
# 
# So in general, each row is an interval of one of these kinds. All other intervals 
# are NaN in that row (so each interval has it's own, sparse column)
# Columns: trial_id	cue	trial_outcome	choice_R1	choice_R2	t0_event_name	t0	x_position	x_alignment	pre_cue_interval	nextto_cue_interval	cue_entry_interval	R1_entry_interval	R2_entry_interval
# So you can see we have the labels, a name of the interval (t0_event_name), the t0 time, and importantly, the x_position 
# we tried to align to (x_alignment) and finally the actual x_position that the 
# unty frame had. So you can see the way it works is for every trial, we check a 
# set of x_alignments (e.g. cue position, R1 position, R2 position) and find the
# ephys time (t0) that corresponds to that x_position. Then we define intervals
# around that t0. The length of the intervals can differ.

# There are also two special intervals, pre_cue_interval, and nextto_cue_interval 
# which are in the same row and short, 200ms,  or 5 ephys bins. The ones above have 
# usually around 30 ephys bins. These were used to train an SVM to decode pre- vs 
# within cue neural activity, first for cue1 and then for cue2 trials. Then I 
# tried to compare these two fits to check if they are orthogonal.


t0_events = analytics.get_analytics('TrialWiseT0Events40ms', session_names=session_names)
t0_events


In [ ]:
behavior = analytics.get_analytics('Behavior40msAligned', session_names=session_names)
fr = analytics.get_analytics('FiringRate40msHz', session_names=session_names)
behavior.iloc[100:200]


In [ ]:
behavior.set_index('to_ephys_timestamp', append=True, inplace=True)
behavior.index = behavior.index.droplevel('entry_id')
behavior

In [ ]:
fr = fr.set_index(["from_ephys_timestamp","to_ephys_timestamp"], append=True)
fr.reset_index(level=(0,1,3,4), inplace=True, drop=True)
fr

In [ ]:

# def select_t0_events(trial_beh):
#     zones = {
#         'cueZone_visible': -120,
#         'cueZone_entry': -80,
#         # 'enter_afterCueZone': 25,
#         # 'enter_reward1Zone': 50,
#         # 'enter_reward2Zone': 170,
#     }
    
#     if trial_beh['trial_id'].iloc[0] == -1:
#         return pd.DataFrame()  # skip ITI
#     base_info = trial_beh[['cue', 'trial_outcome', 'choice_R1', 'choice_R2']].iloc[0].to_dict()
    

#     zone_ts = []
#     for zone_name, zone_x in zones.items():
#         t = np.abs(trial_beh['frame_position']-zone_x).sort_values()
#         # print(zone_name)
#         if t.iloc[0]<3: # a valid location should be within 3 cm of the zone
#             t = t.index[0]
#             x = trial_beh.loc[t, 'frame_position']
#             # print(trial_beh.loc[t])
#             # exit()
#         else:
#             print(f"Warning: No t0 event found for {zone_name}, was {t.iloc[0]} cm away")
#             continue

#         zone_ts.append({**base_info, 't0_event_name': zone_name, 
#                         't0': t, 'x_position': x})
#     zone_ts = pd.DataFrame(zone_ts)
#     return zone_ts

# all_t0_events = []
# for s_id in behavior.index.unique('session_id'):
#     sess_behavior = behavior.xs(s_id, level='session_id')
#     sess_fr = fr.xs(s_id, level='session_id')
#     # make a table where every row is a t0 event, keep info like cue, outcome, choice, trial_id
#     t0_events = sess_behavior.set_index('from_ephys_timestamp', drop=True).groupby('trial_id').apply(select_t0_events, 
#                                                     include_groups=True).reset_index().drop(columns='level_1')
#     # add asession_id to index
#     t0_events['session_id'] = s_id
#     t0_events.set_index(['session_id', 't0'], inplace=True)
    
#     all_t0_events.append(t0_events)
# t0_events = pd.concat(all_t0_events)


In [ ]:
behavior

In [ ]:
# camera = 'bodycam'
# plt.close("all")
# for s_id in t0_events.index.unique('session_id'):
#     if s_id <8:
#         continue
    
#     data = session_modality_from_nas(session_dirs[s_id], f"{camera}_packages")
#     for t0 in t0_events.loc[s_id].index[::19]:
#         print(t0_events.loc[s_id, t0])
        
#         fig, axes = plt.subplots(ncols=6, figsize=(20,3))
#         fig.suptitle(f'Session {s_id}, t0: {t0}', fontsize=16)
#         for i, delta in enumerate(range(-12, 12, 4)):
#             closest_frame = data.iloc[(data[f'{camera}_image_ephys_timestamp'] - t0).abs().argsort()[0] +delta]
#             frame_id = int(closest_frame.loc[f'{camera}_image_id']) 
#             ephys_t = closest_frame.loc[f'{camera}_image_ephys_timestamp'] + (delta * 40000)
            
#             closest_beh_t = (behavior.xs(s_id, level='session_id').loc[:, 'from_ephys_timestamp'] - ephys_t).abs().argsort().values[0]
#             closest_beh_frame = behavior.xs(s_id, level='session_id').iloc[closest_beh_t]
            
#             # print(f"Frame id: {frame_id}, ephys time: {ephys_t}, closest behavior time: {closest_beh_t}, frame position: {closest_beh_frame['frame_position']}")
#             # print(f"Looking for t0: {t0}, closest frame id: {frame_id}, ephys time: {ephys_t}, diff: {ephys_t - t0}")
#             # print(f"Position at t0: {t0_events.loc[(s_id, t0), 'x_position']}")
#             title = f'x: {closest_beh_frame['frame_position']:.2f}cm, \nt_diff: {(ephys_t - t0)/1e6:.3f} us, \ndelta={delta}'
            
#             frame_key = f'frame_{frame_id:06d}'

#             img = session_modality_from_nas(session_dirs[s_id], key=f"{camera}_frames", columns=[frame_key])
#             axes[i].imshow(img[0])
#             axes[i].axis('off')
#             axes[i].set_title(title, fontsize=8)
        
#         plt.tight_layout()
#         plt.show()
#         # break
#     break
        

In [ ]:
data = analytics.get_analytics('SVMCueOutcomeChoicePred', session_names=session_names)
data = data[data.predict_y_name.isin(['pre_vs_in_cue1', 'pre_vs_in_cue2'])]
data

In [ ]:
plt.close("all")
x_modalities = ['HP+mPFC', 'mPFC', 'HP']

no_ephys_s_ids = [0,5,8,17,25,31,29]
ss_cropped_s_ids = [10,24]

# drop sessions with no ephys data
data = data[~data.index.get_level_values('session_id').isin(no_ephys_s_ids)]
data = data[~data.index.get_level_values('session_id').isin(ss_cropped_s_ids)]

for x_modality in x_modalities:
    subset = data[data['X_modality'] == x_modality]
    plt.figure(figsize=(10,4))
    
    # Define colors for each cue
    cue_colors = {'pre_vs_in_cue1': 'orange', 'pre_vs_in_cue2': 'purple'}
    
    for predict_y in ['pre_vs_in_cue1', 'pre_vs_in_cue2']:
        cue_data = subset[subset['predict_y_name'] == predict_y]
        plt.scatter(cue_data.index.unique('session_id'), 
                   cue_data['f1_mean'],
                   color=cue_colors[predict_y],
                   label=f'{predict_y}')
        # add error bars
        for s_id in cue_data.index.unique('session_id'):
             plt.plot([s_id, s_id], 
                      [cue_data.loc[(cue_data.index.get_level_values('session_id') == s_id), 'f1_ci_lower'].values[0],
                       cue_data.loc[(cue_data.index.get_level_values('session_id') == s_id), 'f1_ci_upper'].values[0]],
                      color=cue_colors[predict_y], alpha=0.5)

    # Add reference lines and markers
    plt.scatter(no_ephys_s_ids, [.15]*len(no_ephys_s_ids), color='gray', marker='x', s=50, label='No Ephys Sessions')
    plt.scatter(ss_cropped_s_ids, [.12]*len(ss_cropped_s_ids), color='lightgray', marker='x', s=50, label='SS cropped')
    plt.axvline(x=25.5, color='blue', alpha=.4, linestyle='--', label='Reversal after')
    plt.axvline(x=7.5, color='gray', linestyle='--', label='double rewards before')
    
    plt.title(f'SVM F1 Score over Time - {x_modality}')
    plt.axhline(.5, color='black', linestyle=':', label='Chance Level')
    plt.ylim(.1,.9)
    plt.xlabel('Session ID')
    plt.xticks(np.arange(34))
    plt.ylabel('Before vs within-cue zone SVM decoding F1')
    plt.tight_layout()
    plt.legend()
    plt.show()
    
    # Plot angles between SVM weights
    plt.figure(figsize=(10,4))
    for s_id in subset.index.unique('session_id'):
        cue1_data = subset[(subset.index.get_level_values('session_id') == s_id) & 
                          (subset['predict_y_name'] == 'pre_vs_in_cue1')]
        cue2_data = subset[(subset.index.get_level_values('session_id') == s_id) & 
                          (subset['predict_y_name'] == 'pre_vs_in_cue2')]
        
        if not cue1_data.empty and not cue2_data.empty:
            cue1_w = cue1_data['w'].values[0]
            cue2_w = cue2_data['w'].values[0]
            cos_angle = np.dot(cue1_w, cue2_w) / (np.linalg.norm(cue1_w) * np.linalg.norm(cue2_w))
            angle_deg = np.arccos(np.clip(cos_angle, -1.0, 1.0)) * (180/np.pi)
            plt.scatter(s_id, angle_deg, color='teal')
            
    plt.title(f'Angle between Cue 1 and Cue 2 SVM Weights - {x_modality}')
    plt.axhline(90, color='black', linestyle=':', label='Orthogonal (90 degrees)')
    plt.xlabel('Session ID')
    plt.xticks(np.arange(34))
    plt.ylabel('Angle (degrees)')
    plt.ylim(0,180)
    plt.tight_layout()
    plt.show()

    # Correlation plots
    angles = []
    f1_scores = []
    for s_id in subset.index.unique('session_id'):
        cue1_data = subset[(subset.index.get_level_values('session_id') == s_id) & 
                          (subset['predict_y_name'] == 'pre_vs_in_cue1')]
        cue2_data = subset[(subset.index.get_level_values('session_id') == s_id) & 
                          (subset['predict_y_name'] == 'pre_vs_in_cue2')]
        
        if not cue1_data.empty and not cue2_data.empty:
            cue1_w = cue1_data['w'].values[0]
            cue2_w = cue2_data['w'].values[0]
            cos_angle = np.dot(cue1_w, cue2_w) / (np.linalg.norm(cue1_w) * np.linalg.norm(cue2_w))
            angle_deg = np.arccos(np.clip(cos_angle, -1.0, 1.0)) * (180/np.pi)
            angles.append(angle_deg)
            f1_scores.append(cue1_data['f1_mean'].values[0])

    if angles and f1_scores:
        plt.figure(figsize=(6,4))
        plt.scatter(angles, f1_scores, color='brown')
        m, b = np.polyfit(angles, f1_scores, 1)
        plt.plot(angles, m*np.array(angles) + b, color='orange', linestyle='--', label='Fit Line')
        corr, pval = stats.pearsonr(angles, f1_scores)
        plt.title(f'Correlation between Cue Weight Angle and F1 Score - {x_modality}\n r={corr:.2f}, p={pval:.3f}')
        plt.xlabel('Angle between Cue 1 and Cue 2 Weights (degrees)')
        plt.ylabel('Mean F1 Score')
        plt.ylim(.1,.9)
        plt.tight_layout()
        plt.legend()
        plt.show()

Is it meaningful what the angles are in DG? if they don't decode in these dims, why would these angles be relevant? But why don't they lay around the expected value: 90 dgrees? ALso these angles seems meaningful looking at the correlation with doecoding performance results. I should do a shuffle from which i build a null distribution of angles to expect if there is no structure in the data. 

Something interseting in session 9 to 11 - in both we can decode the cue1 versus not cue1 - but their W are very differnt. Namely the W from 11 onwards coincides with good behavior performance. Cue2 doesn't have that pattern. Their W stays similar. Because also cue2 performance / link to action stays the same

keep in mind what you are plotting here. you random is orthogonal, like on the first 7 sessions. You are not meaninguflly seperating location 1 vs two with your SVM. Your vectors are pretty random and are at 90 degrees. You are comparing pop activity before and after seeing the cue, and then looking at the vector for cue1 and cue2. do they activate in similar direction? Yes. They don't orthogonolize - it t seems rather that they signal relevant object here.

The more aligned the activations when seeing the cue1 and cue2, the higher the decoding accuracy weather we are before, or after the cue. So thie "Some relevant generalcue in fornt of me" dimenions the mPFC shows up again? Is it obvious that alignment between them gives better decoding?

In [ ]:
session_name = '2024-12-09_17-45_rYL006_P1100_LinearTrackStop_26min'

svm_results = analytics.get_analytics('SVMCueOutcomeChoicePred', session_names=session_names)
# svm_results = svm_results[svm_results.predict_y_name == 'cue1_vs_cue2']

# svm_results = analytics.get_analytics('SVMCueOutcomeChoicePred', session_names=session_names)
# svm_results = svm_results[svm_results.predict_y_name == 'trial_outcome']
# svm_results = svm_results[svm_results.predict_y_name == 'choice_R1']
# svm_results = svm_results[svm_results.predict_y_name == 'choice_R2']


svm_results



In [ ]:
session_name = '2024-12-09_17-45_rYL006_P1100_LinearTrackStop_26min'

svm_results = analytics.get_analytics('SVMCueOutcomeChoicePred', session_names=session_names)
svm_results



In [ ]:
svm_results = analytics.get_analytics('SVMCueOutcomeChoicePred2', session_names=session_names)
svm_results
svm_results.xs(15, level='session_id')

In [ ]:
# # TODO duplicated from SVM claulcations, to T0Events calc? would need to incl features such as bin_size etc

# t0_events = analytics.get_analytics('T0Events', session_names=session_names)

# # # make a table where every row is a t0 event, keep info like cue, outcome, choice, trial_id
# cue_visible_t0s = t0_events[t0_events.t0_event_name == 'cueZone_visible']
# cue_entry_t0s = t0_events[t0_events.t0_event_name == 'cueZone_entry']
# # print(cue_visible_t0s)

# timebins_per_fit = 3
# timebins_into_past = 5
# timebins_into_future = 40
# bin_length_us = 40_000  # 40 ms bins

# # Create interval endpoints for each t0 event -200ms to +1600ms
# interval_lefts = cue_visible_t0s.t0 - bin_length_us * timebins_into_past
# interval_rights = cue_visible_t0s.t0 + bin_length_us * timebins_into_future
# cue_visible_t0s['full_interval'] = pd.IntervalIndex.from_arrays(
#     left=interval_lefts,
#     right=interval_rights,
#     closed='right'
# )

# # create before t0 and after t1 inderval windows of 200ms (cue fully invisible vs visible)
# cue_visible_t0s['pre_cue_interval'] = pd.IntervalIndex.from_arrays(
#     left=cue_visible_t0s.t0 - (bin_length_us * 5), # 200 ms before cue visible
#     right=cue_visible_t0s.t0,
#     closed='right'
# )
# cue_visible_t0s['inside_cue_interval'] = pd.IntervalIndex.from_arrays(
#     left=cue_entry_t0s.t0,
#     right=cue_entry_t0s.t0 + (bin_length_us * 5), # 200 ms after cue entry
#     closed='right'
# )

# t0_events = cue_visible_t0s.set_index("trial_id", append=True)
# t0_events = t0_events.droplevel(['paradigm_id', 'animal_id', 'entry_id'])
# t0_events

In [ ]:
mPFC_upper = 106.81
mPFC_lower = 72.81
HP_upper = 123.52
HP_lower = 56.06
HP_mPFC_upper = 105.00
HP_mPFC_lower = 74.63
# insetad do dict
angle_2std_thr = {
    'mPFC': (mPFC_lower, mPFC_upper),
    'HP': (HP_lower, HP_upper),
    'HP+mPFC': (HP_mPFC_lower, HP_mPFC_upper)
}

In [ ]:
t0_events = analytics.get_analytics('TrialWiseT0Events40ms', session_names=session_names)
t0_events.index = t0_events.index.droplevel(['entry_id', 'paradigm_id', 'animal_id'])
t0_events.set_index("trial_id", append=True, inplace=True)
t0_events

In [ ]:
behavior = analytics.get_analytics('Behavior40msAligned', session_names=session_names)
behavior = behavior.reset_index().drop(columns=['paradigm_id', 'animal_id', 'entry_id'])
behavior.index = pd.IntervalIndex.from_arrays(
    left=behavior.to_ephys_timestamp - 40_000,
    right=behavior.to_ephys_timestamp,
    closed='right'
)
behavior.set_index('session_id', append=True, inplace=True) 
behavior

In [ ]:
plt.rcParams.update({
    'font.size': 15,           # general font size
    'axes.titlesize': 15,      # subplot title
    'axes.labelsize': 15,      # x and y labels
    'xtick.labelsize': 15,     # x tick labels
    'ytick.labelsize': 15,     # y tick labels
    'legend.fontsize': 15,     # legend
})

In [ ]:
svm_results_real = analytics.get_analytics('SVMCueOutcomeChoicePred', session_names=session_names)
svm_results_15split = analytics.get_analytics('SVMCueOutcomeChoicePred2', session_names=session_names)
# svm_results = svm_results_15split
svm_results = svm_results_real
# svm_results.xs(15, level='session_id')

In [ ]:
# POSTER PLOTS

def get_session_data(s_id, t0_events, behavior, svm_results, x_modality):
    """Get all relevant data for a single session and modality"""
    # Get SVM results
    X_modality_svm_results = svm_results.loc[
        (svm_results.index.get_level_values('session_id') == s_id) & 
        (svm_results['X_modality'] == x_modality) &
        (svm_results['predict_y_name'] == y_predicted_name) &
        (svm_results['interval_name'].isin(which_intervals))
    ]
    
    # Get behavior trajectories
    all_pos = {}
    print("\nGetting positions data for session:", s_id)
    for which_interval in which_intervals:
        print("Interval ", which_interval)
        # all_pos[which_interval] = []
        trial_poss = []
        for trial_id, full_intrvl in t0_events.loc[s_id, which_interval].dropna().items():

            trial_behavior = behavior.xs(s_id, level='session_id')
            trial_behavior = trial_behavior[trial_behavior['trial_id'] == trial_id]
            dat = trial_behavior.loc[trial_behavior.index.overlaps(full_intrvl)]
            print(dat.shape, end="; ")
            trial_poss.append(dat.frame_position.values)

        # Filter trials with wrong length
        lengths = pd.Series([len(x) for x in trial_poss])
        correct_length = lengths.value_counts().index[0]
        print(f"\n{correct_length=}")
        trial_pos = [pos for pos in trial_poss if len(pos) == correct_length]
        all_pos[which_interval] = np.stack(trial_pos)
        print("SVM data shape: ", X_modality_svm_results[X_modality_svm_results.interval_name == which_interval].shape)
        print()
    return X_modality_svm_results, all_pos, dat.index.mid.values - dat.index.mid[0]

def create_subplot(axes, all_pos, svm_res, title, y_predicted_name):
    """Create a single subplot with trajectories and F1 score coloring"""
    
    # iter intervals
    offset = 0
    xticks = []
    for which_interval in which_intervals:
        # Plot individual trajectories
        this_interval_pos = all_pos[which_interval]
        svm_res_interval = svm_res[svm_res.interval_name == which_interval]

        for pos in this_interval_pos:
            axes[0].plot(svm_res_interval.timebin + offset, pos[1:-1], alpha=0.05, color='k', zorder=4)

        # Plot bin boundaries
        for t in svm_res_interval.timebin:
            axes[0].axvline(x=t-.5 + offset, color='k', linestyle='--', alpha=0.04)

        x_0 = 3+ offset if which_interval == 'cue_entry_interval' else 10+ offset
        xticks.extend([x_0, x_0+25])
        axes[0].axvline(x=x_0, color='k', linestyle='--', alpha=0.4)
        axes[1].axvline(x=x_0, color='k', linestyle='--', alpha=0.4)
        axes[2].axvline(x=x_0, color='k', linestyle='--', alpha=0.4)
        axes[2].axhline(y=x_0, color='k', linestyle='--', alpha=0.4)
        
        scatter = axes[0].scatter(svm_res_interval.timebin + offset, (this_interval_pos).mean(axis=0)[1:-1],
                            c=svm_res_interval.f1_mean,
                            cmap=plt.cm.viridis,
                            norm=Normalize(vmin=0.2, vmax=0.8),
                            alpha=1,
                            s=30,
                            zorder=5)
        
        # plot performance over timepoints
        axes[1].plot(svm_res_interval.timebin + offset, svm_res_interval.f1_mean, color='black', alpha=0.6, zorder=5)
        # draw confidence intervals svm_res.f1_ci_lower, svm_res.f1_ci_upper
        axes[1].fill_between(svm_res_interval.timebin + offset, svm_res_interval.f1_ci_lower, 
                             svm_res_interval.f1_ci_upper, color='black', alpha=0.07,
                             edgecolor=None)
        
        axes[2].axvline(svm_res_interval.timebin.max()+offset +.5, color='white', linewidth=3, alpha=1, zorder=5)
        axes[2].axhline(svm_res_interval.timebin.max()+offset +.5, color='white', linewidth=3, alpha=1, zorder=5)
        offset += svm_res_interval.timebin.max() +1
    
    axes[1].axhline(0.5, color='black', linestyle=':', label='Chance Level')
    axes[1].set_ylabel(f'Decode {y_predicted_name} F1 Score')
    # y grid
    axes[1].yaxis.grid(True, linestyle='--', alpha=0.5)
    axes[1].set_ylim(.1,.9)
    info_str = (f"$n_{{y=0}}={svm_res_interval.n_negatives.values[0]}$, "
                f"$n_{{y=1}}={svm_res_interval.n_positives.values[0]}$")
                # f"n bootstr.: {svm_res_interval.n_iterations.values[0]} ") #, failed: {svm_res_interval.n_success_iterations.values[0]-svm_res_interval.n_iterations.values[0]}, ")
    axes[1].text(0.95, 0.75, info_str, transform=axes[1].transAxes,
        fontsize=10, verticalalignment='bottom', horizontalalignment='right')
        
        
            
    # Add reference lines
    # rectangle for cue zone
    # axes[0].hlines(y=-120, xmin=0, xmax=offset, color='gray', linestyle='--', label='Cue Zone Visible')
    axes[0].fill_betweenx(y=[-80, 0], x1=0, x2=offset, color='black', hatch='/', alpha=0.3, label='Cue Zone')
    axes[0].fill_betweenx(y=[50, 130], x1=0, x2=offset, color='darkgray', alpha=0.6, label='R1 Zone')
    axes[0].fill_betweenx(y=[170, 250], x1=0, x2=offset, color='lightgray', alpha=0.6, label='R2 Zone')

    # Labels
    # axes[0].legend()
    axes[0].set_title(title)
    axes[0].tick_params(labelleft=False, labelbottom=False)

    return scatter, xticks

def plot_session_analysis(s_id, t0_events, behavior, svm_results):
    """Plot complete analysis for one session"""
    fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(7, 6), height_ratios=[1.5, 1, 2], 
                             sharex=True)
    # reduce space between subplots
    plt.subplots_adjust(hspace=0.01, wspace=0.01)
    # y_predicted_name = svm_results['predict_y_name'].values[0] # all the same
    # y_predicted_name = 'cue'
    
    # Plot first row with existing analysis
    # for i, x_modality in enumerate(['mPFC', 'HP']):
    for i, x_modality in enumerate(['mPFC']):
        svm_res, all_pos, all_times = get_session_data(s_id, t0_events, behavior, svm_results, x_modality)
        # f1_scores = [np.nan, *svm_res.f1_mean.to_list(), np.nan]
        scatter, xticks = create_subplot(axes[:,i], all_pos, svm_res, 
                                 f'{y_predicted_name} decoding from \n{x_modality}', y_predicted_name)
        
        
        # Get the timebin values for x-axis alignment
        timebins = np.arange(len(svm_res))
        # Plot heatmap with aligned x-axis
        Ws = np.array(svm_res.w.to_list())
        cos_sim_matrix = np.dot(Ws, Ws.T) / (np.linalg.norm(Ws, axis=1)[:, np.newaxis] * np.linalg.norm(Ws, axis=1)[np.newaxis, :])
        cos_sim_matrix = np.arccos(np.clip(cos_sim_matrix, -1.0, 1.0)) * (180/np.pi)

        high_angles_alpha = ((cos_sim_matrix > angle_2std_thr[x_modality][1]) | (cos_sim_matrix < angle_2std_thr[x_modality][0])).astype(float)
        high_f1_mask = svm_res.f1_mean.values > .55
        high_angles_alpha[high_angles_alpha==False] = .3
        # high_angles_alpha[~high_f1_mask,:] = .3
        # high_angles_alpha[:, ~high_f1_mask] = .3

        # Use pcolormesh instead of imshow for better alignment
        X, Y = np.meshgrid(timebins, timebins)
        im = axes[2,i].pcolormesh(X, Y, cos_sim_matrix, 
                                  alpha=high_angles_alpha,
                                 cmap='bwr_r', vmin=0, vmax=180)
        
        axes[2,i].set_xticks(xticks)
        axes[2,i].set_xticklabels(['t=0' if i%2==0 else '1s' for i in range(len(xticks))])
    
        # Set proper aspect ratio and limits
        # axes[2,i].set_aspect('equal')
        axes[2,i].set_xlim(-0.5, len(timebins)-0.5)
        axes[2,i].set_ylim(-0.5, len(timebins)-0.5)
        axes[2,i].tick_params(labelleft=False)
        # turn off all spines
        [sp.set_visible(False) for sp in axes[0,i].spines.values()]
        [sp.set_visible(False) for sp in axes[1,i].spines.values()]
        [sp.set_visible(False) for sp in axes[2,i].spines.values()]
        
        
    # Add colorbar
    cbar = plt.colorbar(scatter, ax=axes[1,-1])
    cbar.set_ticks([.75, .5, .25])
    cbar.set_label('SVM F1 Score', labelpad=15)
    
    # other colorbar
    cbar2 = plt.colorbar(im, ax=axes[-1,-1])
    cbar2.set_ticks([0, 60, 90, 120, 180])
    cbar2.set_label('Angle between SVM planes (degrees)', labelpad=15)

    plt.suptitle(f'Session {s_id}', fontsize=16, y = .99, )
    plt.tight_layout()
    plt.show()

    fname = os.path.join(output_dir, f'session_S{s_id}_{y_predicted_name}_svm_analysis_S15_Rest.svg')
    fig.savefig(fname, dpi=300)
    #degree sign
    # 45 °


plt.close("all")
y_predicted_name = 'cue'
# which_intervals = ['cue_entry_interval', ]'cue_exit_interval', 'R1_entry_interval', 'R2_entry_interval']
which_intervals = ['cue_entry_interval', 'R2_entry_interval',]

# Main execution
for s_id in svm_results.index.unique('session_id'):
    print(f"Plotting session {s_id}")
    # if s_id in (24,25):
    # if s_id not in (9,10,11,12,13,14):
        # session with cropped SS, skip
    if s_id not in (15,):
        
        continue
    plot_session_analysis(s_id, t0_events, behavior, svm_results)
    # if s_id == 3:

    # break

In [ ]:
trialw_behavior = analytics.get_analytics('BehaviorTrialwise', session_names=session_names[14:15])
trialw_behavior.index = trialw_behavior.index.droplevel(['entry_id', 'paradigm_id', 'animal_id'])
trialw_behavior.set_index("trial_id", append=True, inplace=True)
trialw_behavior.loc[:, 'trial_outcome'] = trialw_behavior.loc[:,'trial_outcome'].clip(0,1)

print(trialw_behavior.cue.value_counts())
print(trialw_behavior.choice_R1.value_counts())
print(trialw_behavior.trial_outcome.value_counts())
# print(trialw_behavior.choice_R2.value_counts())

In [ ]:
trialw_behavior = analytics.get_analytics('BehaviorTrialwise', session_names=session_names[14:15])
trialw_behavior.index = trialw_behavior.index.droplevel(['entry_id', 'paradigm_id', 'animal_id'])
trialw_behavior.set_index("trial_id", append=True, inplace=True)
trialw_behavior.loc[:, 'trial_outcome'] = trialw_behavior.loc[:,'trial_outcome'].clip(0,1)

print(trialw_behavior.cue.value_counts())
print(trialw_behavior.choice_R1.value_counts())
print(trialw_behavior.trial_outcome.value_counts())

# Create visualization of label overlap
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
plt.subplots_adjust(hspace=0.05)  # Reduce space between plots

# Get data
cue_1_trials = trialw_behavior[trialw_behavior.cue == 1]
cue_2_trials = trialw_behavior[trialw_behavior.cue == 2]

# Top plot: Cue distribution (Orange for Cue 1, Purple for Cue 2)
cue_counts = [len(cue_1_trials), len(cue_2_trials)]
axes[0].barh([0], [cue_counts[0]], left=0, color='orange', label='Cue 1', height=0.6)
axes[0].barh([0], [cue_counts[1]], left=cue_counts[0], color='purple', label='Cue 2', height=0.6)
axes[0].set_ylim(-0.5, 0.5)
axes[0].set_xlim(0, sum(cue_counts))
axes[0].set_yticks([0])
axes[0].set_yticklabels(['Cue'])
# axes[0].legend(loc='upper right')
axes[0].set_title('Distribution of Labels Across Trials', fontsize=14, fontweight='bold')
axes[0].text(cue_counts[0]/2, 0, f'n={cue_counts[0]}', ha='center', va='center', fontsize=10, fontweight='bold')
axes[0].text(cue_counts[0] + cue_counts[1]/2, 0, f'n={cue_counts[1]}', ha='center', va='center', fontsize=10, fontweight='bold')

# Middle plot: Choice_R1 distribution split by cue (Dark gray for Stop R1, Gray for Skip R1)
cue1_choice_counts = [
    len(cue_1_trials[cue_1_trials.choice_R1 == False]),    # Skip R1
    len(cue_1_trials[cue_1_trials.choice_R1 == True])  # Stop R1
]
print(cue_1_trials)
print(cue1_choice_counts)
cue2_choice_counts = [
    len(cue_2_trials[cue_2_trials.choice_R1 == False]),    # Skip R1
    len(cue_2_trials[cue_2_trials.choice_R1 == True]),  # Stop R1
]

# Cue 1 side
axes[1].barh([0], [cue1_choice_counts[0]], left=0, color='#404040', label='Stop R1', height=0.6)
axes[1].barh([0], [cue1_choice_counts[1]], left=cue1_choice_counts[0], color='#808080', label='Skip R1', height=0.6)
# Cue 2 side
axes[1].barh([0], [cue2_choice_counts[0]], left=cue_counts[0], color='#404040', height=0.6)
axes[1].barh([0], [cue2_choice_counts[1]], left=cue_counts[0] + cue2_choice_counts[0], color='#808080', height=0.6)

axes[1].set_ylim(-0.5, 0.5)
axes[1].set_xlim(0, sum(cue_counts))
axes[1].set_yticks([0])
axes[1].set_yticklabels(['Choice R1'])
# axes[1].legend(loc='upper right')
axes[1].axvline(cue_counts[0], color='black', linestyle='--', alpha=0.3)

# Add counts
axes[1].text(cue1_choice_counts[0]/2, 0, f'{cue1_choice_counts[0]}', ha='center', va='center', fontsize=9, color='white', fontweight='bold')
axes[1].text(cue1_choice_counts[0] + cue1_choice_counts[1]/2, 0, f'{cue1_choice_counts[1]}', ha='center', va='center', fontsize=9, fontweight='bold')
axes[1].text(cue_counts[0] + cue2_choice_counts[0]/2, 0, f'{cue2_choice_counts[0]}', ha='center', va='center', fontsize=9, color='white', fontweight='bold')
axes[1].text(cue_counts[0] + cue2_choice_counts[0] + cue2_choice_counts[1]/2, 0, f'{cue2_choice_counts[1]}', ha='center', va='center', fontsize=9, fontweight='bold')

# Bottom plot: Trial outcome distribution split by cue (Red for no reward, Green for reward)
cue1_outcome_counts = [
    len(cue_1_trials[cue_1_trials.trial_outcome == 0]),  # No Reward
    len(cue_1_trials[cue_1_trials.trial_outcome == 1])   # Reward
]
cue2_outcome_counts = [
    len(cue_2_trials[cue_2_trials.trial_outcome == 0]),  # No Reward
    len(cue_2_trials[cue_2_trials.trial_outcome == 1])   # Reward
]

# Cue 1 side
axes[2].barh([0], [cue1_outcome_counts[0]], left=0, color='red', label='No Reward', height=0.6)
axes[2].barh([0], [cue1_outcome_counts[1]], left=cue1_outcome_counts[0], color='green', label='Reward', height=0.6)
# Cue 2 side
axes[2].barh([0], [cue2_outcome_counts[0]], left=cue_counts[0], color='red', height=0.6)
axes[2].barh([0], [cue2_outcome_counts[1]], left=cue_counts[0] + cue2_outcome_counts[0], color='green', height=0.6)

axes[2].set_ylim(-0.5, 0.5)
axes[2].set_xlim(0, sum(cue_counts))
axes[2].set_yticks([0])
axes[2].set_yticklabels(['Outcome'])
axes[2].legend(loc='upper right')
axes[2].axvline(cue_counts[0], color='black', linestyle='--', alpha=0.3)
axes[2].set_xlabel('Number of Trials', fontsize=12)

# Add counts
axes[2].text(cue1_outcome_counts[0]/2, 0, f'{cue1_outcome_counts[0]}', ha='center', va='center', fontsize=9, color='white', fontweight='bold')
axes[2].text(cue1_outcome_counts[0] + cue1_outcome_counts[1]/2, 0, f'{cue1_outcome_counts[1]}', ha='center', va='center', fontsize=9, fontweight='bold')
axes[2].text(cue_counts[0] + cue2_outcome_counts[0]/2, 0, f'{cue2_outcome_counts[0]}', ha='center', va='center', fontsize=9, color='white', fontweight='bold')
axes[2].text(cue_counts[0] + cue2_outcome_counts[0] + cue2_outcome_counts[1]/2, 0, f'{cue2_outcome_counts[1]}', ha='center', va='center', fontsize=9, fontweight='bold')


plt.tight_layout()
plt.show()
fname = os.path.join(output_dir, f'trial_label_distribution.svg')
fig.savefig(fname, dpi=300)

In [ ]:
trialw_behavior = analytics.get_analytics('BehaviorTrialwise', session_names=session_names)
trialw_behavior.index = trialw_behavior.index.droplevel(['entry_id', 'paradigm_id', 'animal_id'])
trialw_behavior.set_index("trial_id", append=True, inplace=True)
s15_cue = trialw_behavior.loc[15, 'cue'] -1
s15_choiceR1 = trialw_behavior.loc[15, 'choice_R1']
s15_choiceR2 = trialw_behavior.loc[15, 'choice_R2']
s15_outcome = trialw_behavior.loc[15, 'trial_outcome']
# s15_cue
trialw_behavior
session_names[15]

In [ ]:
svm_results_real = analytics.get_analytics('SVMCueOutcomeChoicePred', session_names=session_names)
svm_results_15split = analytics.get_analytics('SVMCueOutcomeChoicePred2', session_names=session_names)
svm_results = svm_results_15split
# svm_results = svm_results_real
# svm_results.xs(15, level='session_id')

In [ ]:
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report, balanced_accuracy_score

def calc_f1_score(y_true, y_pred, trial_subset=slice(None)):
    """Calculate F1 score given true and predicted labels"""
    f1_aggr = classification_report(y_true[trial_subset].values, y_pred[trial_subset], output_dict=True, zero_division=0)['macro avg']['f1-score']
    return f1_aggr
    
def get_session_data(s_id, t0_events, behavior, svm_results, x_modality):
    """Get all relevant data for a single session and modality"""
    # Get SVM results
    X_modality_svm_results = svm_results.loc[
        (svm_results.index.get_level_values('session_id') == s_id) & 
        (svm_results['X_modality'] == x_modality) &
        (svm_results['predict_y_name'] == y_predicted_name) &
        (svm_results['interval_name'].isin(which_intervals))
    ]
    
    # Get behavior trajectories
    all_pos = {}
    print("\nGetting positions data for session:", s_id)
    for which_interval in which_intervals:
        print("Interval ", which_interval)
        # all_pos[which_interval] = []
        trial_poss = []
        for trial_id, full_intrvl in t0_events.loc[s_id, which_interval].dropna().items():

            trial_behavior = behavior.xs(s_id, level='session_id')
            trial_behavior = trial_behavior[trial_behavior['trial_id'] == trial_id]
            dat = trial_behavior.loc[trial_behavior.index.overlaps(full_intrvl)]
            print(dat.shape, end="; ")
            trial_poss.append(dat.frame_position.values)

        # Filter trials with wrong length
        lengths = pd.Series([len(x) for x in trial_poss])
        correct_length = lengths.value_counts().index[0]
        print(f"\n{correct_length=}")
        trial_pos = [pos for pos in trial_poss if len(pos) == correct_length]
        all_pos[which_interval] = np.stack(trial_pos)
        print("SVM data shape: ", X_modality_svm_results[X_modality_svm_results.interval_name == which_interval].shape)
        print()
    return X_modality_svm_results, all_pos, dat.index.mid.values - dat.index.mid[0]

def create_subplot(axes, all_pos, svm_res, title, y_predicted_name):
    """Create a single subplot with trajectories and F1 score coloring"""
    
    # iter intervals
    offset = 0
    for which_interval in which_intervals:
        # Plot individual trajectories
        this_interval_pos = all_pos[which_interval]
        svm_res_interval = svm_res[svm_res.interval_name == which_interval]

        for pos in this_interval_pos:
            axes[0].plot(svm_res_interval.timebin + offset, pos[1:-1], alpha=0.05, color='k', zorder=4)

        # Plot bin boundaries
        for t in svm_res_interval.timebin:
            axes[0].axvline(x=t-.5 + offset, color='k', linestyle='--', alpha=0.04)

        x_0 = 3+ offset if which_interval == 'cue_entry_interval' else 10+ offset
        axes[0].axvline(x=x_0, color='red', linestyle='--', alpha=0.4, 
                   label='SVM Fit Point' if offset==3 else None)
        scatter = axes[0].scatter(svm_res_interval.timebin + offset, (this_interval_pos).mean(axis=0)[1:-1],
                            # c=svm_res_interval.f1_mean,
                            c=svm_res_interval.f1_mean,
                            cmap=plt.cm.viridis,
                            norm=Normalize(vmin=0.2, vmax=0.8),
                            alpha=1,
                            s=30,
                            zorder=5)
        
        # plot performance over timepoints
        # axes[1].plot(svm_res_interval.timebin + offset, svm_res_interval.f1_mean, color='blue')
        axes[1].plot(svm_res_interval.timebin + offset, svm_res_interval.f1_mean, color='blue')
        # draw confidence intervals svm_res.f1_ci_lower, svm_res.f1_ci_upper
        axes[1].fill_between(svm_res_interval.timebin + offset, svm_res_interval.f1_ci_lower, 
                             svm_res_interval.f1_ci_upper, color='blue', alpha=0.07)
        
        axes[2].axvline(svm_res_interval.timebin.max()+offset, color='white', linewidth=3, alpha=1, zorder=5)
        axes[2].axhline(svm_res_interval.timebin.max()+offset, color='white', linewidth=3, alpha=1, zorder=5)
        offset += svm_res_interval.timebin.max() +1
      
        
    axes[1].axhline(0.5, color='black', linestyle=':', label='Chance Level')
    axes[1].set_title(f'F1 Score over Time, different windows')
    axes[1].set_ylabel('F1 Score')
    axes[1].set_ylim(.1,.9)
    info_str = (f"{y_predicted_name}:\n$n_{{y=0}}={svm_res_interval.n_negatives.values[0]}$,\n"
                f"n bootstr.: {svm_res_interval.n_iterations.values[0]}, failed: {svm_res_interval.n_success_iterations.values[0]}, "
                f"$n_{{y=1}}={svm_res_interval.n_positives.values[0]}$")
    axes[1].text(0.95, 0.75, info_str, transform=axes[1].transAxes,
        fontsize=10, verticalalignment='bottom', horizontalalignment='right')
        
        
            
    # Add reference lines
    # rectangle for cue zone
    axes[0].hlines(y=-120, xmin=0, xmax=offset, color='gray', linestyle='--', label='Cue Zone Visible')
    axes[0].fill_betweenx(y=[-80, 25], x1=0, x2=offset, color='black', hatch='/', alpha=0.3, label='Cue Zone')
    axes[0].fill_betweenx(y=[50, 120], x1=0, x2=offset, color='lightgray', alpha=0.6, label='R1 Zone')
    axes[0].fill_betweenx(y=[170, 240], x1=0, x2=offset, color='darkgray', alpha=0.6, label='R2 Zone')

    # Labels
    axes[0].legend()
    axes[0].set_title(title)
    axes[0].tick_params(labelleft=False, labelbottom=False)

    return scatter

def plot_session_analysis(s_id, t0_events, behavior, svm_results):
    """Plot complete analysis for one session"""
    fig, axes = plt.subplots(nrows=3, ncols=2, figsize=(20, 10), height_ratios=[3, 1, 3], 
                             sharex=True)
    # y_predicted_name = svm_results['predict_y_name'].values[0] # all the same
    # y_predicted_name = 'cue'
    
    # Plot first row with existing analysis
    for i, x_modality in enumerate(['mPFC', 'HP+mPFC']):
        svm_res, all_pos, all_times = get_session_data(s_id, t0_events, behavior, svm_results, x_modality)
        # f1_scores = [np.nan, *svm_res.f1_mean.to_list(), np.nan]
        
        # # f1 score for first half of trials
        # new_f1_scores = []
        # # first half slice
        # print(len(s15_cue))
        # first_half = slice(0, len(s15_cue)//3)
        # second_half = slice(len(s15_cue)//3, None)
        # for idx, row in svm_res.iterrows():
        #     newf1 = calc_f1_score(
        #         y_true=s15_cue,
        #         y_pred=row['predictions'],
        #         # trial_subset=first_half,
        #         trial_subset=second_half
        #     )
        #     new_f1_scores.append(newf1)
        #     print(f"Old F1: {row['f1_mean']:.3f}, New F1: {newf1:.3f}, error : {row['f1_aggr'] - newf1:.3f}, should be {row['aggr_error']:.3f}")
        # svm_res['f1_aggr'] = new_f1_scores
    
        scatter = create_subplot(axes[:,i], all_pos, svm_res, 
                                 f'{y_predicted_name} decoding from \n{x_modality}', y_predicted_name)
    
        # Get the timebin values for x-axis alignment
        timebins = np.arange(len(svm_res))
        # Plot heatmap with aligned x-axis
        Ws = np.array(svm_res.w.to_list())
        cos_sim_matrix = np.dot(Ws, Ws.T) / (np.linalg.norm(Ws, axis=1)[:, np.newaxis] * np.linalg.norm(Ws, axis=1)[np.newaxis, :])
        cos_sim_matrix = np.arccos(np.clip(cos_sim_matrix, -1.0, 1.0)) * (180/np.pi)

        high_angles_alpha = ((cos_sim_matrix > angle_2std_thr[x_modality][1]) | (cos_sim_matrix < angle_2std_thr[x_modality][0])).astype(float)
        high_angles_alpha[high_angles_alpha==False] = .3
        
        # Use pcolormesh instead of imshow for better alignment
        X, Y = np.meshgrid(timebins, timebins)
        im = axes[2,i].pcolormesh(X, Y, cos_sim_matrix, 
                                  alpha=high_angles_alpha,
                                 cmap='bwr_r', vmin=0, vmax=180)
        
        # Set proper aspect ratio and limits
        # axes[2,i].set_aspect('equal')
        axes[2,i].set_xlim(-0.5, len(timebins)-0.5)
        axes[2,i].set_ylim(-0.5, len(timebins)-0.5)
        
        # turn off all spines
        [sp.set_visible(False) for sp in axes[0,i].spines.values()]
        [sp.set_visible(False) for sp in axes[1,i].spines.values()]
        [sp.set_visible(False) for sp in axes[2,i].spines.values()]
        
        
    # # Add colorbar
    # cbar = plt.colorbar(scatter, ax=axes[-1,0])
    # cbar.set_label('SVM F1 Score', labelpad=15)
    
    # # other colorbar
    # cbar2 = plt.colorbar(im, ax=axes[-1,1])
    # cbar2.set_label('Angle between SVM Weights (degrees)', labelpad=15)

    plt.suptitle(f'Session {s_id}', fontsize=16, y = .99, )
    plt.tight_layout()
    plt.show()

    fname = os.path.join(output_dir, f'session_S{s_id}_{y_predicted_name}_svm_analysis.png')
    fig.savefig(fname, dpi=300)


plt.close("all")
y_predicted_name = 'cue'
# which_intervals = ['cue_entry_interval', ]'cue_exit_interval', 'R1_entry_interval', 'R2_entry_interval']
which_intervals = ['cue_entry_interval',  'R2_entry_interval']

# Main execution
for s_id in svm_results.index.unique('session_id'):
    print(f"Plotting session {s_id}")
    # if s_id in (24,25):
    if s_id not in (15,):
        # session with cropped SS, skip
        continue
    y_predicted_name = 'cue'
    plot_session_analysis(s_id, t0_events, behavior, svm_results)
    y_predicted_name = 'choice_R1'
    plot_session_analysis(s_id, t0_events, behavior, svm_results)
    y_predicted_name = 'choice_R2'
    plot_session_analysis(s_id, t0_events, behavior, svm_results)
    # if s_id == 3:
    break

In [ ]:
# y_predicted_name = 'cue'
y_predicted_name = 'choice_R1'
which_intervals = ['cue_entry_interval', 'R1_entry_interval',]
x_modal = 'mPFC'
fname = f'avg_f1_vs_session_{y_predicted_name}_{x_modal}.svg'
# s_ids = [9,10,11,12,13,14]
s_ids = [6,7,9,10,11,12,13,14]

avg_f1 = []
for s_id in svm_results.index.unique('session_id'):
    if s_id not in s_ids:
        continue
    s_svm_results = svm_results.xs(s_id, level='session_id')
    s_svm_results = s_svm_results[(s_svm_results['predict_y_name'] == y_predicted_name) &
                              (s_svm_results['X_modality'] == x_modal) &
                              (s_svm_results['interval_name'].isin(which_intervals))]
    avg_f1.append((s_id, s_svm_results.f1_mean.mean()))

avg_f1 = np.array(avg_f1)

plt.figure(figsize=(4,3))
# Color points by F1 score using viridis colormap scaled from 0.2 to 0.8
scatter = plt.scatter(avg_f1[:,0], avg_f1[:,1], 
                     c=avg_f1[:,1],
                     cmap='viridis',
                     norm=Normalize(vmin=0.2, vmax=0.8),
                     s=100,
                     edgecolors='black',
                     linewidths=0.5)

# Linear fit
m, b = np.polyfit(avg_f1[:,0], avg_f1[:,1], 1)
plt.plot(avg_f1[:,0], m*avg_f1[:,0] + b, color='gray', linestyle='--', linewidth=2)
plt.title(y_predicted_name)

# Correlation
corr, pval = stats.pearsonr(avg_f1[:,0], avg_f1[:,1])
plt.text(0.95, 0.1, f'r={corr:.2f}\np={pval:.3f} *', transform=plt.gca().transAxes,
         verticalalignment='bottom', horizontalalignment='right')

plt.xlabel('Session ID')
plt.ylabel('Average F1 Score')
plt.ylim(.2,.8)
plt.yticks([.25, .5, .6, .75])
plt.gca().yaxis.grid(True, linestyle='--', alpha=0.5)
plt.xticks(s_ids)
plt.xlim(min(s_ids)-.5, max(s_ids)+.5)
plt.xlabel('Session')
plt.axhline(.5, color='black', linestyle=':')
# no spines except left
[sp.set_visible(False) for sp in plt.gca().spines.values() if sp != plt.gca().spines['left']]

# # Add colorbar
# cbar = plt.colorbar(scatter)
# cbar.set_ticks([0.25, 0.5, 0.75])
# cbar.set_label('F1 Score', labelpad=15)

plt.tight_layout()
plt.show()

plt.savefig(os.path.join(output_dir, fname), dpi=300)

S15: choice R2 onle becomes decodable in R2, or right before it. This w is not showing up before. 

Something obviou, but worth stating. You should correlate decoding perfoance with behavioral performance - as in cue1 vs cue 2 distinguished -> correct choice. 

ALso, selcect one W vecter an project activation onto it - check if decoder focused on subset of data, eg it was fit do perform well in win trials, and Cue1vsCue2 is not seperated in lose trials. ALso, super interesting to follow these over the whole task. This also helps to see if these projections emerge? die out? over a session? Can we see these state transitions?

One stary can be S9-S15 - then explain collase, and reversal

Add anew thing to trialwise behavior called near threshold stopping, illustrating when "choice precision" was the issue. THis will inflate double stopping fo course, but still intersting, and potitanlly good alternative plot to show. 

Reminder: use these plots above cue1vs2 to determine the best window for decoding before vs within cue. right now -200 from visible, and +200ms after entry


Ok Adding R1 and R2 and outcome works. Now there is sooo many things to look at. At that is just the alignment to the beginning. 

Trial outcome deconding up to S20.  picture is mixed here, sometimes outcomes can be decoded, but more in specifcal sessions. S11-15 are somewhat aligned over sessions, but it looks weaker than the cue axis. It would be very relevant to comparre this axis angle with cue1vs2. In S12-13 the outcome is decoded quite consistently inside the cue region, with alignmet of the w. In following sessions, there is less alignmet, more patchy phases.

CHoice R1 is weakly decodable in 9-15 window, but w are not very aligned except S14+15 (maybe realted to expoert perfomance?)

these interseting w that we see, can we nail down more details on them? How much variance did they used to exaplain in the pop? Or are they "new" - on vs off manifold learning



In [ ]:
svm_results

In [ ]:
from scipy.ndimage import gaussian_filter

plt.close("all")
y_predicted_name = 'choice_R1'
which_intervals = ['cue_entry_interval', 'R1_entry_interval']
session_subset = [6,7,9,10,11,12,13,14]

for x_modal in ['mPFC', 'HP+mPFC']:
    plt.figure(figsize=(20,20))
    svm_res = svm_results[(svm_results['X_modality'] == x_modal) &
                          (svm_results['predict_y_name'] == y_predicted_name) &
                          (svm_results['interval_name'].isin(which_intervals))
                          ].set_index('timebin', append=True)
    svm_res.index = svm_res.index.droplevel(("paradigm_id", "animal_id", "entry_id"))
    svm_res = svm_res.loc[svm_res.index.get_level_values('session_id').isin(session_subset)]

    Ws = np.array(svm_res.w.to_list())
    cos_sim_matrix = np.dot(Ws, Ws.T) / (np.linalg.norm(Ws, axis=1)[:, np.newaxis] * np.linalg.norm(Ws, axis=1)[np.newaxis, :])
    cos_sim_matrix = np.arccos(np.clip(cos_sim_matrix, -1.0, 1.0)) * (180/np.pi)

    # Apply Gaussian blur
    cos_sim_matrix_blurred = gaussian_filter(cos_sim_matrix, sigma=0)

    high_angles_alpha = ((cos_sim_matrix_blurred > angle_2std_thr[x_modal][1]) | 
                         (cos_sim_matrix_blurred < angle_2std_thr[x_modal][0])).astype(float)
    high_f1 = svm_res.f1_mean.values > .55
    high_angles_alpha[~high_f1,:] = .3
    high_angles_alpha[:, ~high_f1] = .3
    high_angles_alpha[high_angles_alpha==False] = .3

    # Use pcolormesh instead of imshow for better alignment
    X, Y = np.meshgrid(np.arange(cos_sim_matrix.shape[1]), np.arange(cos_sim_matrix.shape[0]))
    im = plt.pcolormesh(X, Y, cos_sim_matrix_blurred, 
                                alpha=high_angles_alpha,
                                cmap='bwr_r', vmin=0, vmax=180)
    
    plt.colorbar(im, label='Angle (degrees)')
    plt.title(f'Cosine Similarity of SVM Weights - {x_modal} - {y_predicted_name}')
    plt.xlabel('Session-Timebin Index') 
    plt.ylabel('Session-Timebin Index')
    
    # draw every session-timebin tick where timebin == 20 (middle of cue visible period)
    ticks = [t if t[1] == 20 else None for t in svm_res.index]
    new_s_ids = np.diff(svm_res.index.get_level_values('session_id')) != 0
    new_s_ids = np.insert(new_s_ids, 0, True)
    new_s_ids = np.arange(len(svm_res))[new_s_ids]
    
    # draw axh and v lines
    for t in range(len(svm_res)):
        if ticks[t] is not None:
            plt.axhline(y=t-20, color='black', linestyle='--', linewidth=.5, alpha=0.1)
            plt.axvline(x=t-20, color='black', linestyle='--', linewidth=.5, alpha=0.1)
    all_lbls = []
    for ns in new_s_ids:
        lw = 2 if svm_res.index[ns][0] in (20,26) else 0.5
        plt.axhline(y=ns-0.5, color='gray', linestyle='-', linewidth=lw, alpha=0.2)
        plt.axvline(x=ns-0.5, color='gray', linestyle='-', linewidth=lw, alpha=0.2)
        lbl = f"S{svm_res.index[ns][0]:02d}"
        plt.text(ns, -5, lbl, 
                 verticalalignment='bottom', horizontalalignment='left', fontsize=6)
        plt.text(-5, ns+5, lbl,
                 verticalalignment='top', horizontalalignment='right', fontsize=6)
        all_lbls.append(lbl)
        
    plt.yticks(new_s_ids, all_lbls)
    plt.xticks(new_s_ids, all_lbls)
    plt.tick_params(axis='both', which='both', labelleft=False, labelright=True, right=True)
    plt.tight_layout()
    plt.show()
    
    # save figure
    fname = os.path.join(output_dir, f'cosine_similarity_{y_predicted_name}_{x_modal}_blurred.png')
    plt.savefig(fname, dpi=300)
    # break
    
    plt.figure(figsize=(6,6))
    print(f"shape: {cos_sim_matrix.shape}")
    high_f1_data = cos_sim_matrix[high_f1][:, high_f1].flatten()
    print(f"n high F1: {len(high_f1_data)}")
    low_f1_data = cos_sim_matrix[~high_f1][:, ~high_f1].flatten()
    print(f"n low F1: {len(low_f1_data)}")
    
    plt.hist(high_f1_data[high_f1_data>2], bins=50, color='green', edgecolor='black', alpha=0.5 , density=True)
    plt.hist(low_f1_data[low_f1_data>2], bins=50, color='gray', edgecolor='black', alpha=0.5   , density=True)
    # t test
    t_stat, p_val = stats.ttest_ind(high_f1_data, low_f1_data, equal_var=False)
    plt.text(0.95, 0.95, f't={t_stat:.2f}\np={p_val:.3e}', transform=plt.gca().transAxes,
                verticalalignment='top', horizontalalignment='right')
    plt.title(f'high F1 - Cosine Similarity Angles - {x_modal} - {y_predicted_name}')
    plt.xlabel('Angle (degrees)')
    plt.ylabel('Count')
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
from scipy.ndimage import gaussian_filter
from scipy import stats

plt.close("all")
y_predicted_names = ['cue', 'choice_R1']
which_intervals = ['cue_entry_interval', 'R1_entry_interval']
session_subset = [9, 11, 12, 13, 14]
thr = .55

low_f1_col = '#445e8aff'
high_f1_col = '#9bd15aff'

def plot_cross_predictor_comparison(x_modal, y_predicted_names, which_intervals, session_subset, thr, svm_results):
    """Plot cross-predictor comparison for a single brain region"""
    # Create figure with 6 columns and 2 rows
    fig, axes = plt.subplots(figsize=(15, 8), ncols=6, nrows=2, 
                            gridspec_kw={'height_ratios': [6, 1.5]})
    
    # Get SVM results
    svm_res = svm_results[(svm_results['X_modality'] == x_modal) &
                          (svm_results['predict_y_name'].isin(y_predicted_names)) &
                          (svm_results['interval_name'].isin(which_intervals))
                          ].set_index('timebin', append=True)
    svm_res.index = svm_res.index.droplevel(("paradigm_id", "animal_id", "entry_id"))
    svm_res = svm_res.loc[svm_res.index.get_level_values('session_id').isin(session_subset)]

    # Initialize arrays for storing results
    cue_high_f1_angles = np.zeros((len(session_subset), len(session_subset)))
    cue_low_f1_angles = np.zeros((len(session_subset), len(session_subset)))
    choice_high_f1_angles = np.zeros((len(session_subset), len(session_subset)))
    choice_low_f1_angles = np.zeros((len(session_subset), len(session_subset)))
    cross_predictor_high_f1_angles = np.zeros((len(session_subset), len(session_subset)))
    cross_predictor_low_f1_angles = np.zeros((len(session_subset), len(session_subset)))
    
    # Also store std for error bars
    cue_high_f1_angles_std = np.zeros((len(session_subset), len(session_subset)))
    cue_low_f1_angles_std = np.zeros((len(session_subset), len(session_subset)))
    choice_high_f1_angles_std = np.zeros((len(session_subset), len(session_subset)))
    choice_low_f1_angles_std = np.zeros((len(session_subset), len(session_subset)))
    cross_predictor_high_f1_angles_std = np.zeros((len(session_subset), len(session_subset)))
    cross_predictor_low_f1_angles_std = np.zeros((len(session_subset), len(session_subset)))
    
    # Store raw angle distributions for statistical testing
    cue_high_f1_angles_raw = {}
    cue_low_f1_angles_raw = {}
    choice_high_f1_angles_raw = {}
    choice_low_f1_angles_raw = {}
    cross_predictor_high_f1_angles_raw = {}
    cross_predictor_low_f1_angles_raw = {}
    
    # Compute within-predictor similarities for cue
    for i, s_id in enumerate(session_subset):
        for j, s_id_2 in enumerate(session_subset):
            s_svm_res = svm_res[(svm_res['predict_y_name'] == 'cue')].xs(s_id, level='session_id')
            s2_svm_res = svm_res[(svm_res['predict_y_name'] == 'cue')].xs(s_id_2, level='session_id')
            
            s_Ws = np.array(s_svm_res.w.to_list())
            s2_Ws = np.array(s2_svm_res.w.to_list())
            
            # Compute cosine similarity
            cos_sim_matrix = np.dot(s_Ws, s2_Ws.T) / (np.linalg.norm(s_Ws, axis=1)[:, np.newaxis] * 
                                                       np.linalg.norm(s2_Ws, axis=1)[np.newaxis, :])
            cos_sim_matrix = np.arccos(np.clip(cos_sim_matrix, -1.0, 1.0)) * (180/np.pi)
            
            # High F1 mask
            s_high_f1 = s_svm_res.f1_mean.values > thr
            s2_high_f1 = s2_svm_res.f1_mean.values > thr
            
            high_f1_angles = cos_sim_matrix[np.ix_(s_high_f1, s2_high_f1)]
            low_f1_angles = cos_sim_matrix[np.ix_(~s_high_f1, ~s2_high_f1)]
            
            cue_high_f1_angles[i, j] = np.mean(high_f1_angles) if high_f1_angles.size > 0 else np.nan
            cue_low_f1_angles[i, j] = np.mean(low_f1_angles) if low_f1_angles.size > 0 else np.nan
            cue_high_f1_angles_std[i, j] = np.std(high_f1_angles) if high_f1_angles.size > 0 else np.nan
            cue_low_f1_angles_std[i, j] = np.std(low_f1_angles) if low_f1_angles.size > 0 else np.nan
            
            # Store raw distributions for last session comparison
            if j == len(session_subset) - 1:
                cue_high_f1_angles_raw[i] = high_f1_angles.flatten() if high_f1_angles.size > 0 else np.array([])
                cue_low_f1_angles_raw[i] = low_f1_angles.flatten() if low_f1_angles.size > 0 else np.array([])
    
    # Compute within-predictor similarities for choice_R1
    for i, s_id in enumerate(session_subset):
        for j, s_id_2 in enumerate(session_subset):
            s_svm_res = svm_res[(svm_res['predict_y_name'] == 'choice_R1')].xs(s_id, level='session_id')
            s2_svm_res = svm_res[(svm_res['predict_y_name'] == 'choice_R1')].xs(s_id_2, level='session_id')
            
            s_Ws = np.array(s_svm_res.w.to_list())
            s2_Ws = np.array(s2_svm_res.w.to_list())
            
            # Compute cosine similarity
            cos_sim_matrix = np.dot(s_Ws, s2_Ws.T) / (np.linalg.norm(s_Ws, axis=1)[:, np.newaxis] * 
                                                       np.linalg.norm(s2_Ws, axis=1)[np.newaxis, :])
            cos_sim_matrix = np.arccos(np.clip(cos_sim_matrix, -1.0, 1.0)) * (180/np.pi)
            
            # High F1 mask
            s_high_f1 = s_svm_res.f1_mean.values > thr
            s2_high_f1 = s2_svm_res.f1_mean.values > thr
            
            high_f1_angles = cos_sim_matrix[np.ix_(s_high_f1, s2_high_f1)]
            low_f1_angles = cos_sim_matrix[np.ix_(~s_high_f1, ~s2_high_f1)]
            
            choice_high_f1_angles[i, j] = np.mean(high_f1_angles) if high_f1_angles.size > 0 else np.nan
            choice_low_f1_angles[i, j] = np.mean(low_f1_angles) if low_f1_angles.size > 0 else np.nan
            choice_high_f1_angles_std[i, j] = np.std(high_f1_angles) if high_f1_angles.size > 0 else np.nan
            choice_low_f1_angles_std[i, j] = np.std(low_f1_angles) if low_f1_angles.size > 0 else np.nan
            
            # Store raw distributions for last session comparison
            if j == len(session_subset) - 1:
                choice_high_f1_angles_raw[i] = high_f1_angles.flatten() if high_f1_angles.size > 0 else np.array([])
                choice_low_f1_angles_raw[i] = low_f1_angles.flatten() if low_f1_angles.size > 0 else np.array([])
    
    # Compute cross-predictor similarities (cue vs choice_R1)
    for i, s_id in enumerate(session_subset):
        for j, s_id_2 in enumerate(session_subset):
            cue_svm_res = svm_res[(svm_res['predict_y_name'] == 'cue')].xs(s_id, level='session_id')
            choice_svm_res = svm_res[(svm_res['predict_y_name'] == 'choice_R1')].xs(s_id_2, level='session_id')
            
            cue_Ws = np.array(cue_svm_res.w.to_list())
            choice_Ws = np.array(choice_svm_res.w.to_list())
            
            # Compute cross-predictor angles
            cos_sim_matrix = np.dot(cue_Ws, choice_Ws.T) / (np.linalg.norm(cue_Ws, axis=1)[:, np.newaxis] * 
                                                             np.linalg.norm(choice_Ws, axis=1)[np.newaxis, :])
            cos_sim_matrix = np.arccos(np.clip(cos_sim_matrix, -1.0, 1.0)) * (180/np.pi)
            
            # Flip around 90 degrees because cue 0 and choice 1, alignment is blue
            cos_sim_matrix = 180 - cos_sim_matrix
            
            # Masks for high and low F1
            cue_high_f1 = cue_svm_res.f1_mean.values > thr
            choice_high_f1 = choice_svm_res.f1_mean.values > thr
            
            # High F1 cross-predictor angles
            cross_high_angles = cos_sim_matrix[np.ix_(cue_high_f1, choice_high_f1)]
            cross_predictor_high_f1_angles[i, j] = np.mean(cross_high_angles) if cross_high_angles.size > 0 else np.nan
            cross_predictor_high_f1_angles_std[i, j] = np.std(cross_high_angles) if cross_high_angles.size > 0 else np.nan
            
            # Low F1 cross-predictor angles
            cross_low_angles = cos_sim_matrix[np.ix_(~cue_high_f1, ~choice_high_f1)]
            cross_predictor_low_f1_angles[i, j] = np.mean(cross_low_angles) if cross_low_angles.size > 0 else np.nan
            cross_predictor_low_f1_angles_std[i, j] = np.std(cross_low_angles) if cross_low_angles.size > 0 else np.nan
            
            # Store raw distributions for last session comparison
            if j == len(session_subset) - 1:
                cross_predictor_high_f1_angles_raw[i] = cross_high_angles.flatten() if cross_high_angles.size > 0 else np.array([])
                cross_predictor_low_f1_angles_raw[i] = cross_low_angles.flatten() if cross_low_angles.size > 0 else np.array([])

    # Plot all six heatmaps (top row)
    im0 = axes[0, 0].imshow(cue_high_f1_angles, cmap='bwr_r', vmin=60, vmax=120, origin='lower')
    axes[0, 0].set_title(f'Cue Within-Predictor\n(High F1 > {thr})\n{x_modal}')
    axes[0, 0].set_xlabel('Session ID')
    axes[0, 0].set_ylabel('Session ID')
    axes[0, 0].set_xticks(range(len(session_subset)))
    axes[0, 0].set_xticklabels(session_subset)
    axes[0, 0].set_yticks(range(len(session_subset)))
    axes[0, 0].set_yticklabels([f'S{s}' for s in session_subset])
    
    im1 = axes[0, 1].imshow(cue_low_f1_angles, cmap='bwr_r', vmin=60, vmax=120, origin='lower')
    axes[0, 1].set_title(f'Cue Within-Predictor\n(Low F1 ≤ {thr})\n{x_modal}')
    axes[0, 1].set_xlabel('Session ID')
    axes[0, 1].set_ylabel('Session ID')
    axes[0, 1].set_xticks(range(len(session_subset)))
    axes[0, 1].set_xticklabels(session_subset)
    axes[0, 1].set_yticks(range(len(session_subset)))
    axes[0, 1].set_yticklabels([f'S{s}' for s in session_subset])
    
    im2 = axes[0, 2].imshow(choice_high_f1_angles, cmap='bwr_r', vmin=60, vmax=120, origin='lower')
    axes[0, 2].set_title(f'Choice_R1 Within-Predictor\n(High F1 > {thr})\n{x_modal}')
    axes[0, 2].set_xlabel('Session ID')
    axes[0, 2].set_ylabel('Session ID')
    axes[0, 2].set_xticks(range(len(session_subset)))
    axes[0, 2].set_xticklabels(session_subset)
    axes[0, 2].set_yticks(range(len(session_subset)))
    axes[0, 2].set_yticklabels([f'S{s}' for s in session_subset])
    
    im3 = axes[0, 3].imshow(choice_low_f1_angles, cmap='bwr_r', vmin=60, vmax=120, origin='lower')
    axes[0, 3].set_title(f'Choice_R1 Within-Predictor\n(Low F1 ≤ {thr})\n{x_modal}')
    axes[0, 3].set_xlabel('Session ID')
    axes[0, 3].set_ylabel('Session ID')
    axes[0, 3].set_xticks(range(len(session_subset)))
    axes[0, 3].set_xticklabels(session_subset)
    axes[0, 3].set_yticks(range(len(session_subset)))
    axes[0, 3].set_yticklabels([f'S{s}' for s in session_subset])
    
    im4 = axes[0, 4].imshow(cross_predictor_high_f1_angles, cmap='bwr_r', vmin=60, vmax=120, origin='lower')
    axes[0, 4].set_title(f'Cross-Predictor Angles\n(Cue vs Choice_R1, High F1)\n{x_modal}')
    axes[0, 4].set_xlabel('Session ID (Choice_R1)')
    axes[0, 4].set_ylabel('Session ID (Cue)')
    axes[0, 4].set_xticks(range(len(session_subset)))
    axes[0, 4].set_xticklabels(session_subset)
    axes[0, 4].set_yticks(range(len(session_subset)))
    axes[0, 4].set_yticklabels([f'S{s}' for s in session_subset])
    
    im5 = axes[0, 5].imshow(cross_predictor_low_f1_angles, cmap='bwr_r', vmin=60, vmax=120, origin='lower')
    axes[0, 5].set_title(f'Cross-Predictor Angles\n(Cue vs Choice_R1, Low F1)\n{x_modal}')
    axes[0, 5].set_xlabel('Session ID (Choice_R1)')
    axes[0, 5].set_ylabel('Session ID (Cue)')
    axes[0, 5].set_xticks(range(len(session_subset)))
    axes[0, 5].set_xticklabels(session_subset)
    axes[0, 5].set_yticks(range(len(session_subset)))
    axes[0, 5].set_yticklabels([f'S{s}' for s in session_subset])
    
    # Plot line plots (bottom row) - ROTATED with sessions on y-axis
    y_sessions = range(len(session_subset))
    
    # Helper function to add significance stars
    def add_significance_stars(ax, y_sessions, high_angles_raw, low_angles_raw, x_pos):
        for i in y_sessions:
            if i in high_angles_raw and i in low_angles_raw:
                high_data = high_angles_raw[i]
                low_data = low_angles_raw[i]
                if len(high_data) > 1 and len(low_data) > 1:
                    _, p_val = stats.ttest_ind(high_data, low_data, equal_var=False)
                    if p_val < (0.05/len(y_sessions)):  # Bonferroni correction
                        ax.text(x_pos, i, '*', ha='left', va='center', fontsize=16, fontweight='bold')
    
    # Plot 0: Cue - both high and low F1 (ROTATED)
    axes[1, 0].plot(cue_high_f1_angles[:, -1], y_sessions, 'o-', color=high_f1_col, linewidth=2, label=f'High F1 > {thr}')
    axes[1, 0].errorbar(cue_high_f1_angles[:, -1], y_sessions, xerr=cue_high_f1_angles_std[:, -1], 
                        fmt='none', ecolor=high_f1_col, alpha=0.3, capsize=3)
    axes[1, 0].plot(cue_low_f1_angles[:, -1], y_sessions, 'o-', color=low_f1_col, linewidth=2, label=f'Low F1 ≤ {thr}')
    axes[1, 0].errorbar(cue_low_f1_angles[:, -1], y_sessions, xerr=cue_low_f1_angles_std[:, -1], 
                        fmt='none', ecolor=low_f1_col, alpha=0.3, capsize=3)
    add_significance_stars(axes[1, 0], y_sessions, cue_high_f1_angles_raw, cue_low_f1_angles_raw, 108)
    axes[1, 0].set_xlabel('Angle (°)')
    axes[1, 0].set_ylabel('Session ID')
    axes[1, 0].set_yticks(y_sessions)
    axes[1, 0].set_yticklabels([s for s in session_subset])
    axes[1, 0].set_xlim(70, 100)
    axes[1, 0].set_xticks([80,90])
    axes[1, 0].axvline(90, color='gray', linestyle='--', alpha=0.5)
    axes[1, 0].grid(True, alpha=0.3)
    # axes[1, 0].legend()
    axes[1, 0].spines['top'].set_visible(False)
    axes[1, 0].spines['right'].set_visible(False)
    axes[1, 0].spines['bottom'].set_visible(False)
    
    # Hide plot 1
    axes[1, 1].axis('off')
    
    # Plot 2: Choice - both high and low F1 (ROTATED)
    axes[1, 2].plot(choice_high_f1_angles[:, -1], y_sessions, 'o-', color=high_f1_col, linewidth=2, label=f'High F1 > {thr}')
    axes[1, 2].errorbar(choice_high_f1_angles[:, -1], y_sessions, xerr=choice_high_f1_angles_std[:, -1], 
                        fmt='none', ecolor=high_f1_col, alpha=0.3, capsize=3)
    axes[1, 2].plot(choice_low_f1_angles[:, -1], y_sessions, 'o-', color=low_f1_col, linewidth=2, label=f'Low F1 ≤ {thr}')
    axes[1, 2].errorbar(choice_low_f1_angles[:, -1], y_sessions, xerr=choice_low_f1_angles_std[:, -1], 
                        fmt='none', ecolor=low_f1_col, alpha=0.3, capsize=3)
    add_significance_stars(axes[1, 2], y_sessions, choice_high_f1_angles_raw, choice_low_f1_angles_raw, 108)
    axes[1, 2].set_xlabel('Angle (°)')
    axes[1, 2].set_ylabel('Session ID')
    axes[1, 2].set_yticks(y_sessions)
    axes[1, 2].set_yticklabels(session_subset)
    axes[1, 2].set_xlim(70, 100)
    axes[1, 2].set_xticks([80,90])
    axes[1, 2].axvline(90, color='gray', linestyle='--', alpha=0.5)
    axes[1, 2].grid(True, alpha=0.3)
    # axes[1, 2].legend()
    axes[1, 2].spines['top'].set_visible(False)
    axes[1, 2].spines['right'].set_visible(False)
    axes[1, 2].spines['bottom'].set_visible(False)
    
    # Hide plot 3
    axes[1, 3].axis('off')
    
    # Plot 4: Cross-predictor - both high and low F1 (ROTATED)
    axes[1, 4].plot(cross_predictor_high_f1_angles[:, -1], y_sessions, 'o-', color=high_f1_col, linewidth=2, label=f'High F1 > {thr}')
    axes[1, 4].errorbar(cross_predictor_high_f1_angles[:, -1], y_sessions, xerr=cross_predictor_high_f1_angles_std[:, -1], 
                        fmt='none', ecolor=high_f1_col, alpha=0.3, capsize=3)
    axes[1, 4].plot(cross_predictor_low_f1_angles[:, -1], y_sessions, 'o-', color=low_f1_col, linewidth=2, label=f'Low F1 ≤ {thr}')
    axes[1, 4].errorbar(cross_predictor_low_f1_angles[:, -1], y_sessions, xerr=cross_predictor_low_f1_angles_std[:, -1], 
                        fmt='none', ecolor=low_f1_col, alpha=0.3, capsize=3)
    add_significance_stars(axes[1, 4], y_sessions, cross_predictor_high_f1_angles_raw, cross_predictor_low_f1_angles_raw, 108)
    axes[1, 4].set_xlabel('Angle (°)')
    axes[1, 4].set_ylabel('Session ID')
    axes[1, 4].set_yticks(y_sessions)
    axes[1, 4].set_yticklabels(session_subset)
    axes[1, 4].set_xlim(70, 100)
    axes[1, 4].set_xticks([80,90])
    axes[1, 4].grid(True, alpha=0.3)
    # axes[1, 4].legend()
    axes[1, 4].spines['top'].set_visible(False)
    axes[1, 4].spines['right'].set_visible(False)
    axes[1, 4].spines['bottom'].set_visible(False)
    
    # Hide plot 5
    # axes[1, 5].axis('off')
    
    plt.tight_layout()
    
    # Save figure
    
    
    # return data
    return cross_predictor_high_f1_angles[:, -1]

# Call the function for each brain region
x = plot_cross_predictor_comparison('mPFC', y_predicted_names, which_intervals, session_subset, thr, svm_results)



s_ids = [9,11,12,13,14]
# these come froma plot above where we chec what propoertion of stop R1 trials coincide with cue=1 trials
print(x)
cov = [.64,.6,.8,.68,.75]
# x = np.arange(len(s_ids))
plt.scatter(x, cov, edgecolor=high_f1_col, s=100, color='gray', linewidths=1.5)
plt.ylim(.5,1)
plt.xticks((90,80))
plt.xlim(70, 100,)
plt.grid(linestyle='--', alpha=.5)
[sp.set_visible(False) for sp in plt.gca().spines.values()]

# Linear fit
m, b = np.polyfit(x, cov, 1)
plt.plot(x, m*x + b, color='gray', linestyle='--', linewidth=2)
# test correlation p value
corr, pval = stats.pearsonr(x, cov)
plt.title(f'r={corr:.2f}p={pval:.3f} Trial overlap', )
# tick right
plt.tick_params(axis='y', which='both', labelleft=False, labelright=True, right=True, left=False, )
plt.xlabel('Cross-Predictor Angle (degrees)')
plt.show()

fname = os.path.join(output_dir, f'cross_predictor-session_comparison_mPFC.svg')
print(fname)
plt.savefig(fname, dpi=300)

In [ ]:
ensambles = analytics.get_analytics('ConcatenatedEnsambles40ms', session_names=session_names)
ensambles

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def calc_PCA(fr):
    # Identify silent and active neurons
    silent_neuron_rows = fr.index[fr.sum(axis=1) == 0]
    active_neuron_rows = fr.index[fr.sum(axis=1) > 0]

    # Z-score normalization per active neuron
    fr_z = fr.loc[active_neuron_rows].apply(lambda x: (x - x.mean()) / x.std(), axis=1)

    # Correlation matrix and eigendecomposition
    corr_matrix = np.corrcoef(fr_z.values)
    eigenvals, eigenvecs = np.linalg.eigh(corr_matrix)

    # Sort descending by eigenvalue
    idx = np.argsort(eigenvals)[::-1]
    eigenvals = eigenvals[idx]
    eigenvecs = eigenvecs[:, idx]

    # Insert zeros for silent neurons
    new_eigenvecs = []
    for eigenvec in eigenvecs.T:
        full_vec = np.zeros(fr.shape[0])
        full_vec[active_neuron_rows] = eigenvec
        new_eigenvecs.append(full_vec)
    new_eigenvecs.extend([np.zeros(fr.shape[0])] * len(silent_neuron_rows))
    eigenvecs = np.array(new_eigenvecs).T
    eigenvals = np.concatenate([eigenvals, np.zeros(len(silent_neuron_rows))])

    # Restore full neuron matrix shape
    fr_z = fr_z.reindex(fr.index, fill_value=0)

    # Project data onto eigenvectors (optional verification)
    proj_data = np.dot(fr_z.values.T, eigenvecs)
    print('--PCA--')
    print("Projected data shape:", proj_data.shape)
    # var
    var_per_dim = np.var(proj_data, axis=0)
    print("Variance per dimension shape:", var_per_dim.shape)
    print("Variance per dimension (first 10):", var_per_dim[:10])
    print("Sum of variances:", var_per_dim.sum())
    print('----')
    

    return eigenvecs, eigenvals


# ===========================================================
# === PCA CALCULATION =======================================
# ===========================================================

plt.close("all")
y_predicted_names = ['cue', 'trial_outcome', 'choice_R1', 'choice_R2']

for x_modal in ['HP+mPFC', 'mPFC', 'HP']:
    fig, axes = plt.subplots(ncols=len(y_predicted_names), nrows=2, figsize=(20, 20), 
                             height_ratios=[5, 1],
                             sharex=True, )#sharey=True)
    
    for col_idx, (y_predicted_name, ax) in enumerate(zip(y_predicted_names, axes.T)):
        # ===========================================================
        # === DATA PREPARATION ======================================
        # ===========================================================
        svm_res = svm_results[(svm_results['X_modality'] == x_modal) &
                              (svm_results['interval_name'].isin(['cue_entry_interval'])) &
                            (svm_results['predict_y_name'] == y_predicted_name)
                            ].set_index('timebin', append=True)
        svm_res.index = svm_res.index.droplevel(("paradigm_id", "animal_id", "entry_id"))
        # svm_res = svm_res.loc[:20]  # First 20 sessions
        # svm_res = svm_res.loc[14:14]  # First 20 sessions
        
        all_vars = []
        all_s_ids = []
        
        # Project data for each session
        for s_id in svm_results.index.unique('session_id'):
            if s_id > 16:
            # if s_id in (24,25):
                continue
            all_s_ids.append(s_id)
            
            # Select data based on modality
            if x_modal == 'HP+mPFC':
                data = fr.loc[s_id:s_id]
            elif x_modal == 'mPFC':
                data = fr.iloc[:, 20:].loc[s_id:s_id]
            elif x_modal == 'HP':
                data = fr.iloc[:, :20].loc[s_id:s_id]
            eigenvecs, eigenvals = calc_PCA(data.T.reset_index(drop=True))

            # Normalize data
            data = data.apply(lambda x: (x - x.mean()) / x.std(), axis=0).fillna(0)

            # Project onto SVM weights
            Ws = np.array(svm_res.w.to_list()).T  # shape: neurons x (sessions * timebins SVM  W vectors)
            Ws = Ws / np.linalg.norm(Ws, axis=0, keepdims=True)
            print(f"Data shape for session {s_id}: {data.shape}, Ws shape: {Ws.shape}")
            projected_data = data @ Ws  # shape: timepoints x (sessions * timebins)
            print(f"Projected data shape for session {s_id}: {projected_data.shape}")
            var_per_dim = np.var(projected_data, axis=0) / data.shape[1]  # variance per SVM weight dimension
            print(f"Variance per dim shape for session {s_id}: {var_per_dim.shape}, sum: {var_per_dim.sum()}, first 10: {var_per_dim[:10]}")
            all_vars.append(var_per_dim)
            print()
            print()
            print()
        
        all_vars = np.array(all_vars).T  # shape: (sessions * timebins) x sessions
        
        # subtract first session variance
        # all_vars = all_vars - all_vars[:, np.newaxis, 0]
        thr = .55
        # print(all_vars)
        high_f1_vars = all_vars[svm_res.f1_mean.values > thr, :]
        low_f1_vars = all_vars[svm_res.f1_mean.values <= thr, :]
        
        # ax[1].plot(low_f1_vars.T, color='blue', alpha=0.1, )
        # stds in title
        ax[1].set_title(f'Standard Deviation of Variance across Sessions\n'
                        f'High F1 Std: ±{high_f1_vars.flatten().std()*100:.3f}%, '
                        f'Low F1 Std: ±{low_f1_vars.flatten().std()*100:.3f}%')
        ax[1].set_xlabel('Sessions')

        # fill between
        x = np.arange(all_vars.shape[1]) + 0.5
        for svm_w_var_expl in high_f1_vars:
            ax[1].plot(x, svm_w_var_expl, color='green', alpha=0.4, )
        ax[1].fill_between(x,
                        high_f1_vars.mean(axis=0) - high_f1_vars.std(axis=0),
                        high_f1_vars.mean(axis=0) + high_f1_vars.std(axis=0),
                        color='green', alpha=0.3, label=f'F1 > {thr}, Std: {high_f1_vars.std():.2f}')
        ax[1].plot(x, high_f1_vars.mean(axis=0), color='green', alpha=1, label=f'F1 > {thr}, Std: {high_f1_vars.std():.2f}')

        ax[1].fill_between(x,
                        low_f1_vars.mean(axis=0) - low_f1_vars.std(axis=0),
                        low_f1_vars.mean(axis=0) + low_f1_vars.std(axis=0),
                        color='gray', alpha=0.3, label=f'F1 > {thr}, Std: {low_f1_vars.std():.2f}')
        ax[1].plot(x, low_f1_vars.mean(axis=0), color='gray', alpha=1, label=f'F1 > {thr}, Std: {low_f1_vars.std():.2f}')

        # ax[1].plot(high_f1_vars.std(axis=0)[1:], color='gray', alpha=1, label=f'F1 < 0.6, Std: {high_f1_vars.std():.2f}')
        # ax[1].plot(low_f1_vars.std(axis=0)[1:], color='blue', alpha=1, label=f'F1 >= 0.6, Std: {low_f1_vars.std():.2f}')
        
        # axes[2].hist(high_f1_vars.flatten(), bins=30, color='gray', alpha=0.5, density=True, label='F1 < 0.55')
        # axes[3].hist(low_f1_vars.flatten(), bins=30, color='blue', alpha=0.5, density=True, label='F1 >= 0.55')





    #     # ===========================================================
    #     # === VISUALIZATION =========================================
    #     # ===========================================================
        # all_vars[svm_res.f1_mean.values < thr] = np.nan  # mask low variance values for better color scaling
        
        im = ax[0].pcolormesh(all_vars, cmap='magma_r', )#vmin=.005, vmax=.02)
        
        # Add gridlines for session_id boundaries (using svm_res, not filtered)
        session_changes = np.where(np.diff(svm_res.index.get_level_values('session_id')) != 0)[0]
        for change_idx in session_changes:
            ax[0].axhline(y=change_idx + 1, color='lightgray', linestyle='-', linewidth=1.5, alpha=0.5)
        
        # Add gridlines for interval_name boundaries
        if 'interval_name' in svm_res.columns:
            interval_changes = np.where(np.diff(svm_res['interval_name'].astype('category').cat.codes) != 0)[0]
            for change_idx in interval_changes:
                ax[0].axhline(y=change_idx + 1, color='lightgray', linestyle='--', linewidth=1, alpha=0.3)

        # Add y-axis labels for sessions (using svm_res, not filtered)
        y_labels = []
        y_ticks = []
        prev_session = None
        
        for i, (s_id, timebin) in enumerate(svm_res.index):
            if s_id != prev_session:
                y_ticks.append(i)
                y_labels.append(f'S{s_id}')
                prev_session = s_id

        ax[0].set_yticks(y_ticks)
        ax[0].set_yticklabels(y_labels, fontsize=8)

        # X-axis labels for sessions
        ax[0].set_xticks(np.arange(len(all_s_ids)) + 0.5)
        ax[0].set_xticklabels([f"S{t:02d}" for t in all_s_ids], rotation=90, fontsize=6)

        ax[0].set_title(f'{y_predicted_name}', fontsize=10)
        ax[0].set_xlabel('Session ID', fontsize=8)

        # Only show ylabel on leftmost plot
        if col_idx == 0:
            ax[0].set_ylabel('Timebin Index (per session)', fontsize=8)
    
    fig.suptitle(f'Variance of Projected Data onto SVM Weights - {x_modal}', fontsize=14, y=0.98)
    # # Add single colorbar for the entire figure
    fig.colorbar(im, label='Variance', pad=0.01)
    # fig.suptitle(f'Variance of Projected Data - {x_modal}', fontsize=14, y=0.98)
    plt.tight_layout()
    plt.show()
    plt.legend()

Main message: the edcoding axis is preserved over sessions. ANother way to show this would be to check the performance using a prv timepoint decoder.

Session 9 also decodes cue1vscue2, but not. when within the zone, so are these within zone axis maybe related to reward or action?

tjese are these tiny windows in S14 and S16 where the DG is decoding axis is stable? But the decoding performance is quite low...

It would be great to pin down the point of learing better, or of collapse, within session, from trial to trial - using projection i guess?

Big one: after reversal, the axis doesn't change, but activity becomes anti correlated: ie something flipped. Where you maybe looking at a reward expectation signal, and now the cue no longer predict reward? how quickly is this updated? within session? so exciting! check with behavior! 

In [ ]:
plt.close("all")
which_intervals = ['cue_entry_interval', 'R1_entry_interval',] #'R2_entry_interval']
y_predicted_name = 'cue'

# svm_res
for x_modal in ['HP+mPFC', 'mPFC', 'HP']:
# for x_modal in ['mPFC', 'HP']:
    plt.figure(figsize=(20,20))
    svm_res = svm_results[(svm_results['X_modality'] == x_modal) &
                            (svm_results['predict_y_name'] == y_predicted_name) &
                            (svm_results['interval_name'].isin(which_intervals))
                        ]
    
    svm_res.index = svm_res.set_index('timebin', append=True).index    
    # print(svm_res.index.value_counts().sort_index())
    
    svm_res = svm_res.loc[pd.IndexSlice[:,:,6:14]]
    print(svm_res)
    

    Ws = np.array(svm_res.w.to_list())
    cos_sim_matrix = np.dot(Ws, Ws.T) / (np.linalg.norm(Ws, axis=1)[:, np.newaxis] * np.linalg.norm(Ws, axis=1)[np.newaxis, :])
    # convert to degrees
    cos_sim_matrix = np.arccos(np.clip(cos_sim_matrix, -1.0, 1.0)) * (180/np.pi)
    alpha_mask = ((cos_sim_matrix > angle_2std_thr[x_modal][1]) | (cos_sim_matrix < angle_2std_thr[x_modal][0])).astype(float)
    alpha_mask[alpha_mask==False] = .3
    
    im = plt.imshow(cos_sim_matrix, cmap='bwr_r', vmin=0, vmax=180, alpha=alpha_mask)
    plt.colorbar(im, label='Angle (degrees)')
    plt.title(f'Cosine Similarity of SVM Weights - {x_modal} - {y_predicted_name} - Intervals: {which_intervals}')
    plt.xlabel('Session-Timebin Index') 
    plt.ylabel('Session-Timebin Index')
    # draw every session-timebin tick where timebin == 20 (middle of cue visible period)
    # ticks = [t if t[1] == 1 else None for t in svm_res.index]
    
    next_s_idx_mask = np.diff(svm_res.index.get_level_values('session_id'))
    print(next_s_idx_mask)
    next_s_idx_mask = np.insert(next_s_idx_mask, 0, 1).astype(bool)
    session_ticks = np.arange(len(svm_res))[next_s_idx_mask]
    plt.yticks(session_ticks, svm_res.index.unique('session_id') ,fontsize=6)
    plt.xticks(session_ticks, svm_res.index.unique('session_id') ,fontsize=6, rotation=90)
    # grid
    plt.grid(which='both', color='black', linestyle='--', alpha=0.4)
    
    # # draw axh and v lines
    # for t in range(len(svm_res)):
    #     if ticks[t] is not None:
    #         plt.axhline(y=t-20, color='black', linestyle='--', alpha=0.1)
    #         plt.axvline(x=t-20, color='black', linestyle='--', alpha=0.1)
    # plt.xticks(np.arange(len(svm_res)), ticks, fontsize=6)
    # plt.yticks(np.arange(len(svm_res)), ticks, fontsize=6)
    
    # tick labels are session_id
    # plt.yticks(np.arange(len(svm_res)), [f"S{t[0]:02d}" if t is not None else None for t in ticks], fontsize=6)
    # plt.xticks(np.arange(len(svm_res)), [f"S{t[0]:02d}" if t is not None else None for t in ticks], fontsize=6, rotation=90)
    
    plt.tight_layout()
    
    plt.show()
    fname = f"{output_dir}/cosine_similarity_{y_predicted_name}_{x_modal}.png"
    plt.savefig(fname, dpi=300)
    break

In [ ]:
ensambles = analytics.get_analytics('ConcatenatedEnsambles40ms', session_names=session_names)

plt.close("all")
which_intervals = ['cue_entry_interval', 'R1_entry_interval','R2_entry_interval']
y_predicted_name = 'cue'

# svm_res
for x_modal in ['HP+mPFC', 'mPFC', 'HP']:
# for x_modal in ['mPFC', 'HP']:
    plt.figure(figsize=(7,20))
    svm_res = svm_results[(svm_results['X_modality'] == x_modal) &
                            (svm_results['predict_y_name'] == y_predicted_name) &
                            (svm_results['interval_name'].isin(which_intervals))
                        ]
    
    svm_res.index = svm_res.set_index('timebin', append=True).index    
    
    Ws = np.array(svm_res.w.to_list())  # Shape: (n_svm_fits, n_neurons)
    print(f"Ws shape: {Ws.shape}")
    print(f"Ensembles shape: {ensambles.shape}")
    
    # Transpose ensembles to get (n_neurons, n_ensembles)
    ensembles_T = ensambles.values  # Shape: (n_neurons, n_ensembles)
    print(f"Ensembles transposed shape: {ensembles_T.shape}")
    
    # Compute cosine similarity between SVM weights (Ws) and ensembles
    # Result shape: (n_svm_fits, n_ensembles)
    cos_sim_matrix = np.dot(Ws, ensembles_T) / (
        np.linalg.norm(Ws, axis=1)[:, np.newaxis] * 
        np.linalg.norm(ensembles_T, axis=0)[np.newaxis, :]
    )
    
    # Convert to degrees
    cos_sim_matrix = np.arccos(np.clip(cos_sim_matrix, -1.0, 1.0)) * (180/np.pi)
    
    print(f"Cosine similarity matrix shape: {cos_sim_matrix.shape}")
   
    # Create alpha mask based on angle thresholds
    alpha_mask = ((cos_sim_matrix > angle_2std_thr[x_modal][1]) | 
                  (cos_sim_matrix < angle_2std_thr[x_modal][0])).astype(float)
    high_f1_mask = svm_res.f1_mean.values > .55
    alpha_mask[alpha_mask==False] = .3

    
    # Use pcolormesh for visualization
    X, Y = np.meshgrid(np.arange(cos_sim_matrix.shape[1]), np.arange(cos_sim_matrix.shape[0]))
    im = plt.pcolormesh(X, Y, cos_sim_matrix, 
                        alpha=alpha_mask,
                        cmap='bwr_r', vmin=0, vmax=180)
    
    plt.colorbar(im, label='Angle (degrees)')
    plt.title(f'Cosine Similarity: SVM Weights vs Ensembles\n{x_modal} - {y_predicted_name}')
    plt.xlabel('Ensemble Index') 
    plt.ylabel('SVM Fit Index (Session-Timebin)')
    
    # Add session boundaries
    next_s_idx_mask = np.diff(svm_res.index.get_level_values('session_id'))
    next_s_idx_mask = np.insert(next_s_idx_mask, 0, 1).astype(bool)
    session_ticks = np.arange(len(svm_res))[next_s_idx_mask]
    plt.yticks(session_ticks, svm_res.index.unique('session_id'), fontsize=6)
    
    # Grid for clarity
    plt.grid(which='both', color='black', linestyle='--', alpha=0.4)
    
    plt.tight_layout()
    plt.show()
    
    fname = f"{output_dir}/cosine_similarity_SVM_vs_ensembles_{y_predicted_name}_{x_modal}.png"
    plt.savefig(fname, dpi=300)
    break

In [ ]:
svm_results
# svm_results = svm_results[svm_results.w.notna()]

choide: In the long run, over all33 sessions, choice represetantions are more stable than outcome and cue. Even after the christmes break

Cue: One of the best ones: learning in S14 shows up in anticorrelation along cue dim decoding dim in R2. Can you somehow distinquish this type of learning from an orthogonolization? In January, the pattern isn't this nice. 


R2 choice representations are very stable overall, even over christmas.nd cue and outcome almost no long long termstability

control: check for the choice axis you find how well it predict velocity

In [ ]:
svm_results = svm_results_15split
# svm_results = svm_results_real
svm_results = svm_results[svm_results.w.notna()].drop(['cue', 'comparison', 'n', 'name'], axis=1)

In [ ]:
plt.close("all")
which_intervals = ['cue_entry_interval', 'R1_entry_interval', 'R2_entry_interval']
# which_intervals = ['cue_entry_interval', 'R2_entry_interval']
which_pred_y = ['cue', 'choice_R2']

# svm_res
for s_id in svm_results.index.unique('session_id'):
    # if s_id in (24,25):
    if s_id not in [15]:
        continue
    for x_modal in ['HP+mPFC', 'mPFC']:
        # Get and preprocess data
        print(svm_results)
        svm_res = svm_results.loc[pd.IndexSlice[:, : , s_id]]
        svm_res = svm_res[(svm_res['X_modality'] == x_modal) &
                         (svm_res['interval_name'].isin(which_intervals)) &
                         (svm_res['predict_y_name'].isin(which_pred_y))
                        ].reset_index()
        
        print(svm_res)
        svm_res.loc[svm_res['predict_y_name'].isin(['choice_R1']), 'w'] *= -1 
        # invert weights for easier interpretation
        # Cue1 -> 0 Cue2 -> 1
        # stopR1 -> 0 skipR1 -> 1 # needed to be flipped, trained other way round
        # skipR2 -> 0 stopR2 -> 1  # keep
        # outcome 0 -> 0 reward 1 -> 1 # keep
        
        
        # Reorder by intervals and prediction types
        svm_res.set_index(['predict_y_name', 'interval_name', 'timebin'], inplace=True, append=False)
        svm_res = svm_res.reindex(pd.MultiIndex.from_product(
            [which_pred_y, which_intervals, sorted(svm_res.index.unique('timebin'))]))
        print(svm_res.isna().sum())
        # print(svm_res.isna().sum())
        svm_res = svm_res.dropna() # some intervals have fewer timebins than others
        # print(svm_res)
        
        
        # Create numeric mapping for the categories
        pred_y_map = {name: idx for idx, name in enumerate(which_pred_y)}
        interval_map = {name: idx for idx, name in enumerate(which_intervals)}

        # # Get change points
        pred_y_numeric = np.array([pred_y_map[x] for x in svm_res.index.get_level_values(0)])
        interval_numeric = np.array([interval_map[x] for x in svm_res.index.get_level_values(1)])
        pred_y_changes = np.where(np.diff(pred_y_numeric) != 0)[0]
        interval_changes = np.where(np.diff(interval_numeric) != 0)[0]
        all_changes = np.sort(np.unique(np.concatenate([pred_y_changes, interval_changes])))
        change_positions = all_changes + 1


        print(svm_res)
        # print(change_positions)
        # Calculate cosine similarity matrix
        Ws = np.array(svm_res.w.dropna().to_list())
        cos_sim_matrix = np.dot(Ws, Ws.T) / (np.linalg.norm(Ws, axis=1)[:, np.newaxis] * np.linalg.norm(Ws, axis=1)[np.newaxis, :])
        cos_sim_matrix = np.arccos(np.clip(cos_sim_matrix, -1.0, 1.0)) * (180/np.pi)
        
        cos_sim_matrix = cos_sim_matrix.T
        
        # flip over vertical axis for easier interpretation
        # cos_sim_matrix = np.flip(np.flip(cos_sim_matrix, axis=1), axis=0)
        

        # high_angle_mask = ((cos_sim_matrix > angle_2std_thr[x_modal][1]) | (cos_sim_matrix < angle_2std_thr[x_modal][0]))
        # good_pred_mask = (svm_res.f1_mean > 0.55).values
        # alpha_mask = high_angle_mask & (good_pred_mask[:, np.newaxis] & good_pred_mask[np.newaxis, :])
        # alpha_mask = alpha_mask.astype(float)
        # alpha_mask[alpha_mask==False] = .3
        
        high_angles_alpha = ((cos_sim_matrix > angle_2std_thr[x_modal][1]) | 
                         (cos_sim_matrix < angle_2std_thr[x_modal][0])).astype(float)
        high_f1 = svm_res.f1_mean.values > .55
        # high_angles_alpha[~high_f1,:] = .3
        # high_angles_alpha[:, ~high_f1] = .3
        # high_angles_alpha[high_angles_alpha==False] = .3
        
        
        # cos_sim_matrix[~good_pred_mask, :] = np.nan
        # cos_sim_matrix[:, ~good_pred_mask] = np.nan

        # Create figure and plot
        fig = plt.figure(figsize=(10,10))
        plt.pcolormesh(cos_sim_matrix, cmap='bwr_r', vmin=0, vmax=180, alpha=high_angles_alpha)
        # plt.colorbar(label='Angle (degrees)')
        
        # Draw grid lines and labels
        for i, pos in enumerate(change_positions):
            if pos in pred_y_changes + 1:  # Major changes (prediction type)
                plt.axhline(y=pos, color='black', linestyle='-', alpha=0.3)
                plt.axvline(x=pos, color='black', linestyle='-', alpha=0.3)
                
            elif pos in interval_changes + 1:  # Minor changes (intervals)
                plt.axhline(y=pos, color='gray', linestyle='--', alpha=0.2)
                plt.axvline(x=pos, color='gray', linestyle='--', alpha=0.2)
                
        plt.title(f'Session {s_id} - Cosine Similarity of SVM Weights\n{x_modal}', pad=20, fontsize=14)
        
        # Remove default ticks
        plt.xticks([])
        plt.yticks([])
        
        # Adjust layout
        plt.tight_layout()
        plt.subplots_adjust(left=0.15, bottom=0.1, top=0.9)
        
        # fname = f"cosine_similarity_S{s_id}_{x_modal}.png"
        # plt.savefig(os.path.join(output_dir, fname), dpi=300)
        
        plt.show()
    #     break
    # break

In [ ]:
plt.close("all")
which_intervals = ['cue_entry_interval', 'R1_entry_interval']
y_predicted_name = 'cue'

# svm_res
# for x_modal in ['HP+mPFC', 'mPFC', 'HP']:
for x_modal in ['mPFC', 'HP', 'HP+mPFC']:
    plt.figure(figsize=(5,5))
    svm_res = svm_results[(svm_results['X_modality'] == x_modal) &
                            (svm_results['predict_y_name'] == 'cue')
                        ].loc[:8]
    
    Ws = np.array(svm_res.w.to_list())
    cos_sim_matrix = np.dot(Ws, Ws.T) / (np.linalg.norm(Ws, axis=1)[:, np.newaxis] * np.linalg.norm(Ws, axis=1)[np.newaxis, :])
    # convert to degrees
    degrees = (np.arccos(np.clip(cos_sim_matrix, -1.0, 1.0)) * (180/np.pi)).flatten()
    
    hist = plt.hist(degrees, bins=80, color='gray')
    # draw in 2 std dev lines
    mean_angle = np.nanmean(degrees)
    std_angle = np.nanstd(degrees)
    print(mean_angle + 2*std_angle, x_modal)
    print(mean_angle - 2*std_angle, x_modal)
    plt.axvline(mean_angle + 2*std_angle, color='lightblue', linestyle='--', label=f'{mean_angle + 2*std_angle:.2f}°')
    plt.axvline(mean_angle - 2*std_angle, color='lightblue', linestyle='--', label=f'{mean_angle - 2*std_angle:.2f}°')
    plt.title(f'Histogram of Cosine Similarity of SVM Weights - {x_modal} - {y_predicted_name}')
    plt.legend()
    plt.xlabel('Angle (degrees)') 
    plt.ylabel('Count')
    plt.tight_layout()
plt.show()

In [ ]:
fr

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def calc_PCA(fr):
    # Identify silent and active neurons
    silent_neuron_rows = fr.index[fr.sum(axis=1) == 0]
    active_neuron_rows = fr.index[fr.sum(axis=1) > 0]

    # Z-score normalization per active neuron
    fr_z = fr.loc[active_neuron_rows].apply(lambda x: (x - x.mean()) / x.std(), axis=1)

    # Correlation matrix and eigendecomposition
    corr_matrix = np.corrcoef(fr_z.values)
    eigenvals, eigenvecs = np.linalg.eigh(corr_matrix)

    # Sort descending by eigenvalue
    idx = np.argsort(eigenvals)[::-1]
    eigenvals = eigenvals[idx]
    eigenvecs = eigenvecs[:, idx]

    # Insert zeros for silent neurons
    new_eigenvecs = []
    for eigenvec in eigenvecs.T:
        full_vec = np.zeros(fr.shape[0])
        full_vec[active_neuron_rows] = eigenvec
        new_eigenvecs.append(full_vec)
    new_eigenvecs.extend([np.zeros(fr.shape[0])] * len(silent_neuron_rows))
    eigenvecs = np.array(new_eigenvecs).T
    eigenvals = np.concatenate([eigenvals, np.zeros(len(silent_neuron_rows))])

    # Restore full neuron matrix shape
    fr_z = fr_z.reindex(fr.index, fill_value=0)

    # Project data onto eigenvectors (optional verification)
    _ = np.dot(fr_z.values.T, eigenvecs)

    return eigenvecs, eigenvals


# ===========================================================
# === PCA CALCULATION =======================================
# ===========================================================
data = (fr.loc[:19])
# drop 10 and 17 due to cropped sessions
data = data.drop(index=[10,17], level='session_id')
print(data)
eigenvecs, eigenvals = calc_PCA(data.T.reset_index(drop=True))

# ===========================================================
# === VISUALIZATION =========================================
# ===========================================================
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

plt.figure(figsize=(6, 10))

# Create colormap normalization for eigenvalues
norm = Normalize(vmin=0.005, vmax=0.02)
cmap = plt.cm.magma_r
sm = ScalarMappable(norm=norm, cmap=cmap)

# Base stacked bar of explained variance (total PCA)
x = -1
bottom = 0
eigenval_sum = np.sum(eigenvals)
print(eigenvals / eigenval_sum)
for i, eigenval in enumerate(eigenvals):
    evr = eigenval / eigenval_sum
    color = cmap(norm(np.clip(evr, 0.005, .02)))
    plt.bar(x, evr, bottom=bottom, edgecolor='gray', color=color)
    bottom += evr

# Per-session explained variance projection
all_s_ids = []
all_pc_exmpl_var = []
for s_id in fr.index.unique('session_id'): #[:21]:
    if s_id in (10, 17):  # skip cropped sessions
        continue
    if s_id >19:
        continue
    all_s_ids.append(s_id)

    s_fr = fr.loc[s_id]
    s_fr_z = s_fr.apply(lambda x: (x - x.mean()) / x.std(), axis=0).fillna(0)

    # Project session data onto PCA eigenvectors
    projected_data = np.dot(s_fr_z.values, eigenvecs)
    pc_var = np.var(projected_data, axis=0)

    bottom = 0
    for i, pv in enumerate(pc_var):
        evr = pv / np.sum(pc_var)
        color = cmap(norm(np.clip(evr, 0.005, .02)))
        
        plt.bar(s_id, evr, bottom=bottom, edgecolor='gray', color=color)
        bottom += evr
        
    all_pc_exmpl_var.append(pc_var / np.sum(pc_var))
    
all_pc_exmpl_var = np.array(all_pc_exmpl_var)
print(all_pc_exmpl_var)
print(all_pc_exmpl_var.shape)

plt.xticks(all_s_ids, [f"S{s_id}" for s_id in all_s_ids])
plt.title('Explained Variance by PCA Components across Sessions')
plt.xlabel('Session ID')
plt.ylabel('Explained Variance')
plt.xticks(rotation=90)

# Add colorbar
cbar = plt.colorbar(sm, ax=plt.gca())
cbar.set_label('Eigenvalue (clipped 0.5-1.5)', rotation=270, labelpad=20)

plt.tight_layout()
plt.show()


plt.figure(figsize=(5,5))
plt.plot(all_s_ids, all_pc_exmpl_var, marker='o', linestyle='-', alpha=0.7)
# plt.plot(all_s_ids, all_pc_exmpl_var -all_pc_exmpl_var[0], marker='o', linestyle='-', alpha=0.7)
plt.title('Distribution of Explained Variance across Sessions')
plt.tight_layout()
plt.xticks(all_s_ids, [f"S{s_id}" for s_id in all_s_ids], rotation=90)
plt.show()